# Perplexity Analysis — TAT German

This notebook runs the full statistical analysis and visualization pipeline.
It loops over three analysis settings (with outliers, without outliers, matched sample)
and produces the same set of plots and statistics for each.

In [3]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
import matplotlib
import statsmodels.api as sm
from scipy.stats import spearmanr
matplotlib.use('Agg')
from functions.analysis.utils import check_confounder_neutrality, plot_neutrality_check
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Make sure project root is on the path
sys.path.insert(0, os.path.abspath('.'))

from services.analysis_service import analysis_service, load_metadata, build_person_df
from services.plotting_service import plot_setting, plot_cross_setting, plot_advanced, plot_matched_sample_balance
from functions.analysis.statistics import *
from functions.plotting.plots import (
    plot_group_boxplot,
    plot_roc_curves,
    plot_metrics_heatmap,
    plot_raw_vs_residualized,
    format_and_save_clinical_correlations,
)
DIAG_ORDER = ["HC","MDD", "BD", "SSD"]
DIAG_COLORS = {"HC": "#3498DB", "SSD": "#E74C3C", "MDD": "#F39C12", "BD": "#9B59B6"}

## 1. Configuration

In [4]:
# Paths
TYPE = "prompt_healthy" #prompt_neutral prompt_no_prompt prompt_healthy prompt_sick
SENTENCE_CSV = f"data/02_perplexities/data/02_perplexities_vllm/leo-mistral-hessianai-7b-chat-instruct/{TYPE}/TAT_german_utterances_with_perplexities_{TYPE.split('_', 1)[1]}.csv"
METADATA_CSV = "data/00_raw_data/speech_metadata.csv"
FIGURE_DIR = "figures"
TABLE_DIR = "tables"
METADATA_COLS = [
    "file_id", "diagnosis", "sex", "age", "education", "comorbidity",
    "GAF", "verbal_IQ", "TLI_Impoverished", "TLI_Disorganization", "TLI_Non_specific",
]

# Model prefix for column names
MODEL = "qwen"

# Primary perplexity variable
#qwen_loss_mean_wins, 
RAW_LM_VAR =f"{MODEL}_loss_median" #f"{MODEL}_loss_median"
PPL_VAR = f"{MODEL}_loss_median"
WC_VAR = f"{MODEL}_total_words"

PREPRROCSSING = "with_outliers" # "with_outliers" or "without_outliers"

# Clinical variable setup: small set for plots, larger set for correlation tables
extra_clinical_vars = [
    # Add any additional metadata columns you want to test here, e.g. "PANSS_Total"
    # "PANSS_Total",
    # "CDSS_Total",
]

# Keep full candidate list here; we will validate against df.columns after df is built
clinical_vars_corr = list(dict.fromkeys(CLINICAL_VARS + extra_clinical_vars))
clinical_vars = [v for v in ["GAF", "TLI_Impoverished"] if v in clinical_vars_corr]

print(f"Clinical vars for plots (requested): {clinical_vars}")
print(f"Clinical vars for correlation tables (requested, pre-validation): {len(clinical_vars_corr)}")

Clinical vars for plots (requested): ['GAF', 'TLI_Impoverished']
Clinical vars for correlation tables (requested, pre-validation): 19


## 2. Load data & prepare three analysis settings

In [5]:
match_vars = [f"{MODEL}_avg_words", "education"]


# Then change the global definition:
CONFOUNDER_VARS = ["sex_num", "age", f"{MODEL}_avg_words", "education"]
#CONFOUNDER_VARS = ["sex", "age", f"{MODEL}_avg_words", "education"] #"verbal_IQ"
MATCH_K = 2  # number of HCs per patient in per-diagnosis matching
df, settings = analysis_service(
    sentence_csv=SENTENCE_CSV,
    metadata_csv=METADATA_CSV,
    model_name=MODEL,
    match_vars=match_vars,
    match_k=MATCH_K,
    requested_clinical_vars=clinical_vars_corr,
    impute_confounders_cols=CONFOUNDER_VARS,
 )
#dropped IDs on basis of PREPROCESSING
df_base = df.copy()
df = settings[PREPRROCSSING]
print(f"\nDropped {len(df_base) - len(df)} subjects based on preprocessing choice: {PREPRROCSSING}")
df["sex_num"] = df["sex"].map({
    "M": 0, "F": 1, "m": 0, "f": 1,
    "male": 0, "female": 1, "Male": 0, "Female": 1,
})
# Validate requested clinical vars against the actual analysis df (after rename + merge)
missing_corr_vars = [v for v in clinical_vars_corr if v not in df.columns]
clinical_vars_corr = [v for v in clinical_vars_corr if v in df.columns]
clinical_vars = [v for v in clinical_vars if v in clinical_vars_corr]

print(f"\nFull dataset: {len(df)} subjects")
print(f"Patients: {(df['is_patient'] == 1).sum()}, Controls: {(df['is_patient'] == 0).sum()}")
print(f"Clinical vars for correlation tables (available): {len(clinical_vars_corr)}")
if missing_corr_vars:
    print(f"Dropped missing clinical vars: {missing_corr_vars}")
#df to docs
df.to_csv(f"{TABLE_DIR}/full_dataset_for_analysis.csv", index=False)

                 Variable           HC           BD          MDD          SSD       P
                        n          178           27          119           48        
               Age, years  44.4 ± 13.5  44.6 ± 12.2  44.3 ± 13.8  41.2 ± 12.2    0.47
         Education, years   14.6 ± 3.2   13.4 ± 3.1   13.4 ± 3.0   12.0 ± 2.2 < 0.001
                Verbal IQ 116.1 ± 12.8 111.8 ± 13.2 116.4 ± 14.2 111.9 ± 15.1    0.26
            Female, n (%)  114 (64.0%)   17 (63.0%)   74 (62.2%)   18 (37.5%)   0.009
       Words per sentence  20.3 ± 11.1  19.5 ± 13.0   18.9 ± 9.8   15.6 ± 6.4   0.005
      Number of sentences  66.4 ± 26.7  61.9 ± 22.9  66.1 ± 24.9  71.4 ± 36.8    0.90
        Clinical measures                                                            
            HAMD-17 total            —    5.8 ± 5.8    5.6 ± 5.8    9.2 ± 8.4    0.01
               YMRS total            —    5.8 ± 8.9    0.4 ± 1.0    6.0 ± 7.2 < 0.001
                      GAF            —  59.7 ± 13.3  6

In [6]:
# ── Add length-residualized loss features ─────────────────────────

sentence_df = pd.read_csv(SENTENCE_CSV)

required_loss_cols = [
    "file_id",
    "avg_loss_completion",
    "word_count_completion",
    "utterance",
]

missing = [c for c in required_loss_cols if c not in sentence_df.columns]
if missing:
    raise ValueError(f"Missing required columns for residualized loss: {missing}")

sentence_df_resid, length_model = add_length_residualized_loss(sentence_df)

if length_model is not None:
    print("\nSentence-level length residualization model:")
    print(length_model.summary())

agg_loss = aggregate_loss_completion(sentence_df_resid, MODEL)
agg_resid = aggregate_residualized_loss(sentence_df_resid, MODEL)

merge_keys = [
    "file_id",
    f"{MODEL}_total_words",
    f"{MODEL}_avg_words",
    f"{MODEL}_n_sentences",
]

loss_person_df = agg_loss.merge(
    agg_resid,
    on=merge_keys,
    how="outer",
)

loss_person_df = add_winsorized_columns(
    loss_person_df,
    cols=[
        f"{MODEL}_loss_mean",
        f"{MODEL}_loss_median",
        f"{MODEL}_loss_resid_mean",
        f"{MODEL}_loss_resid_median",
    ],
    lower_q=0.01,
    upper_q=0.99,
)

# Only merge columns that are not already in df, except file_id
new_loss_cols = [
    c for c in loss_person_df.columns
    if c == "file_id" or c not in df.columns
]

#preprocess the file_id in the same way as for df, to ensure they match for merging, remove everything after _ and delete any special characters strip whitespaces
loss_person_df["file_id"] = loss_person_df["file_id"].apply(lambda x: str(x).split("_")[0].replace("-", "").replace(" ", "").strip())
df = df.merge(
    loss_person_df[new_loss_cols],
    on="file_id",
    how="left",
)

# Also merge into every setting dataframe
for setting_name in list(settings.keys()):
    settings[setting_name] = settings[setting_name].merge(
        loss_person_df[new_loss_cols],
        on="file_id",
        how="left",
    )

# Sanity check
print("\nAvailable loss columns:")
print([c for c in df.columns if f"{MODEL}_loss" in c])

if PPL_VAR not in df.columns:
    raise ValueError(f"Primary metric {PPL_VAR} not found in df.columns")


Sentence-level length residualization model:
                             OLS Regression Results                            
Dep. Variable:     avg_loss_completion   R-squared:                       0.481
Model:                             OLS   Adj. R-squared:                  0.481
Method:                  Least Squares   F-statistic:                 2.293e+04
Date:                 Sat, 23 May 2026   Prob (F-statistic):               0.00
Time:                         17:39:48   Log-Likelihood:                -40998.
No. Observations:                24758   AIC:                         8.200e+04
Df Residuals:                    24756   BIC:                         8.202e+04
Df Model:                            1                                         
Covariance Type:             nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------

In [7]:
settings.keys()

dict_keys(['with_outliers', 'without_outliers', 'matched', 'matched_without_outliers', 'matched_per_dx', 'matched_per_dx_without_outliers'])

In [8]:
from scipy.stats import spearmanr, pearsonr
tmp = sentence_df_resid[
    ["avg_loss_completion", "avg_loss_length_resid", "word_count_completion"]
].replace([np.inf, -np.inf], np.nan).dropna()

print("Raw loss vs residualized loss")
print("Pearson:", pearsonr(tmp["avg_loss_completion"], tmp["avg_loss_length_resid"]))
print("Spearman:", spearmanr(tmp["avg_loss_completion"], tmp["avg_loss_length_resid"]))

print("\nRaw loss vs word count")
print("Pearson:", pearsonr(tmp["avg_loss_completion"], tmp["word_count_completion"]))
print("Spearman:", spearmanr(tmp["avg_loss_completion"], tmp["word_count_completion"]))

print("\nResidualized loss vs word count")
print("Pearson:", pearsonr(tmp["avg_loss_length_resid"], tmp["word_count_completion"]))
print("Spearman:", spearmanr(tmp["avg_loss_length_resid"], tmp["word_count_completion"]))

Raw loss vs residualized loss
Pearson: PearsonRResult(statistic=np.float64(0.720490220565068), pvalue=np.float64(0.0))
Spearman: SignificanceResult(statistic=np.float64(0.5632903826059306), pvalue=np.float64(0.0))

Raw loss vs word count
Pearson: PearsonRResult(statistic=np.float64(-0.48300035152375104), pvalue=np.float64(0.0))
Spearman: SignificanceResult(statistic=np.float64(-0.6723160201989369), pvalue=np.float64(0.0))

Residualized loss vs word count
Pearson: PearsonRResult(statistic=np.float64(0.13880825361133375), pvalue=np.float64(9.40692078481254e-107))
Spearman: SignificanceResult(statistic=np.float64(0.12229260018289823), pvalue=np.float64(4.103678638181443e-83))


In [9]:
# ── Diagnostic: which variables actually correlate with PPL? ──────
from scipy.stats import spearmanr as _sp, mannwhitneyu
from scipy.stats import pointbiserialr as _pb
def fmt_p(p):
    if p < 0.001:
        return "< 0.001"
    elif p < 0.01:
        return f"{p:.3f}"
    else:
        return f"{p:.2f}"
candidate_vars = CONFOUNDER_VARS
print(f"{'Variable':<25} {'r_all':>8} {'p_all':>10} {'r_HC':>8} {'p_HC':>10} {'r_PAT':>8} {'p_PAT':>10}")
print("-" * 85)

for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1})
    else:
        col = df[var]

    # All subjects
    mask_all = col.notna() & df[PPL_VAR].notna()
    r_all, p_all = _sp(col[mask_all], df[PPL_VAR][mask_all])

    # HC only
    hc = df["is_patient"] == 0
    mask_hc = hc & col.notna() & df[PPL_VAR].notna()
    r_hc, p_hc = _sp(col[mask_hc], df[PPL_VAR][mask_hc])

    # Patients only
    pat = df["is_patient"] == 1
    mask_pat = pat & col.notna() & df[PPL_VAR].notna()
    r_pat, p_pat = _sp(col[mask_pat], df[PPL_VAR][mask_pat])

    sig_all = "***" if p_all < .001 else "**" if p_all < .01 else "*" if p_all < .05 else "ns"
    sig_hc = "***" if p_hc < .001 else "**" if p_hc < .01 else "*" if p_hc < .05 else "ns"
    sig_pat = "***" if p_pat < .001 else "**" if p_pat < .01 else "*" if p_pat < .05 else "ns"

    print(f"{var:<25} {r_all:>+.3f} {p_all:>10.4f} {sig_all:>3}  {r_hc:>+.3f} {p_hc:>10.4f} {sig_hc:>3}  {r_pat:>+.3f} {p_pat:>10.4f} {sig_pat:>3}")

# Also check: do these variables differ between groups?
print(f"\n{'Variable':<25} {'HC mean':>10} {'PAT mean':>10} {'p(MW)':>10} {'sig':>4}")
print("-" * 65)
for var in candidate_vars:
    if var == "sex_num":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1})
        r_pb_all, p_pb_all = _pb(col[mask_all], df[PPL_VAR][mask_all])
        r_pb_hc, p_pb_hc = _pb(col[mask_hc], df[PPL_VAR][mask_hc])
        r_pb_pat, p_pb_pat = _pb(col[mask_pat], df[PPL_VAR][mask_pat])
        
        print(f"\n--- Specific Analysis for SEX (Point-Biserial) ---")
        print(f"ALL: r_pb = {r_pb_all:+.3f}, p = {p_pb_all:.4f}")
        print(f"HC:  r_pb = {r_pb_hc:+.3f}, p = {p_pb_hc:.4f}")
        print(f"PAT: r_pb = {r_pb_pat:+.3f}, p = {p_pb_pat:.4f}")
        print(f"--------------------------------------------------\n")
    else:
        col = df[var]
    hc_vals = col[df["is_patient"] == 0].dropna()
    pat_vals = col[df["is_patient"] == 1].dropna()
    stat, p_mw = mannwhitneyu(hc_vals, pat_vals, alternative="two-sided")
    sig = "***" if p_mw < .001 else "**" if p_mw < .01 else "*" if p_mw < .05 else "ns"
    print(f"{var:<25} {hc_vals.mean():>10.3f} {pat_vals.mean():>10.3f} {p_mw:>10.4f} {sig:>4}")

import pandas as pd

# ── Table 1: Spearman correlations with PPL ──────
rows_corr = []
for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1})
    else:
        col = df[var]

    mask_all = col.notna() & df[PPL_VAR].notna()
    r_all, p_all = _sp(col[mask_all], df[PPL_VAR][mask_all])

    hc = df["is_patient"] == 0
    mask_hc = hc & col.notna() & df[PPL_VAR].notna()
    r_hc, p_hc = _sp(col[mask_hc], df[PPL_VAR][mask_hc])

    pat = df["is_patient"] == 1
    mask_pat = pat & col.notna() & df[PPL_VAR].notna()
    r_pat, p_pat = _sp(col[mask_pat], df[PPL_VAR][mask_pat])


    rows_corr.append({
        "Variable": var,
        "ρ_all": round(r_all, 2), "P_all": fmt_p(p_all),
        "ρ_HC": round(r_hc, 2),  "P_HC": fmt_p(p_hc),
        "ρ_PAT": round(r_pat, 2), "P_PAT": fmt_p(p_pat),
    })

df_corr = pd.DataFrame(rows_corr)

# ── Table 2: Group comparisons (Mann-Whitney U) ──────
rows_mw = []
for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1})
    else:
        col = df[var]
    hc_vals = col[df["is_patient"] == 0].dropna()
    pat_vals = col[df["is_patient"] == 1].dropna()
    stat, p_mw = mannwhitneyu(hc_vals, pat_vals, alternative="two-sided")

    rows_mw.append({
        "Variable": var,
        "HC mean": round(hc_vals.mean(), 2),
        "HC SD": round(hc_vals.std(), 2),
        "PAT mean": round(pat_vals.mean(), 2),
        "PAT SD": round(pat_vals.std(), 2),
        "U": round(stat, 1),
        "P": fmt_p(p_mw),
    })

df_mw = pd.DataFrame(rows_mw)

# ── Save both to one Excel file with two sheets ──────
with pd.ExcelWriter(f"{FIGURE_DIR}/confounder/supplementary_confounder_tables.xlsx") as writer:
    df_corr.to_excel(writer, sheet_name="Spearman_correlations", index=False)
    df_mw.to_excel(writer, sheet_name="Group_comparisons", index=False)

print("Saved: supplementary_confounder_tables.xlsx")

Variable                     r_all      p_all     r_HC       p_HC    r_PAT      p_PAT
-------------------------------------------------------------------------------------
sex_num                   -0.118     0.0227   *  -0.084     0.2645  ns  -0.133     0.0660  ns
age                       +0.187     0.0003 ***  +0.150     0.0462   *  +0.224     0.0017  **
qwen_avg_words            -0.785     0.0000 ***  -0.766     0.0000 ***  -0.792     0.0000 ***
education                 -0.353     0.0000 ***  -0.346     0.0000 ***  -0.300     0.0000 ***

Variable                     HC mean   PAT mean      p(MW)  sig
-----------------------------------------------------------------

--- Specific Analysis for SEX (Point-Biserial) ---
ALL: r_pb = -0.118, p = 0.0232
HC:  r_pb = -0.068, p = 0.3676
PAT: r_pb = -0.140, p = 0.0529
--------------------------------------------------

sex_num                        0.640      0.562     0.1229   ns
age                           44.360     43.541     0.5037  

In [10]:
# ── Correlation: matching variables & confounders vs perplexity ────────────
from scipy.stats import spearmanr as _sp

plot_vars = match_vars + CONFOUNDER_VARS
plot_vars = [v if v != "sex" else "sex_num" for v in match_vars + CONFOUNDER_VARS]
nice_names = {
    f"{MODEL}_avg_words": "Avg words per sentence",
    "sex": "Sex (0=m, 1=f)",
    "age": "Age",
}

n_vars = len(plot_vars)
fig, axes = plt.subplots(1, n_vars, figsize=(5.5 * n_vars, 4.5))
if n_vars == 1:
    axes = [axes]

for ax, var in zip(axes, plot_vars):
    sub = df[[var, PPL_VAR]].dropna()
    is_cat = sub[var].nunique() <= 5
    x_vals = sub[var] + (np.random.uniform(-0.15, 0.15, len(sub)) if is_cat else 0)
    ax.scatter(x_vals, sub[PPL_VAR], alpha=0.45, s=18, color="#5B7DB1", edgecolors="none")
    # regression line + Spearman r
    r, p = _sp(sub[var], sub[PPL_VAR])
    z = np.polyfit(sub[var].astype(float), sub[PPL_VAR], 1)
    xs = np.linspace(sub[var].min(), sub[var].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", lw=1.5, alpha=0.7)
    ax.set_xlabel(nice_names.get(var, var), fontsize=11)
    ax.set_ylabel("Median perplexity" if ax == axes[0] else "")
    sig = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
    ax.set_title(f"r = {r:.3f}, p = {p:.4f} {sig}", fontsize=10)
    ax.grid(alpha=0.2)

fig.suptitle(f"Matching & confounder variables vs {PPL_VAR}", fontsize=13, y=1.02)
fig.tight_layout()
os.makedirs(os.path.join(FIGURE_DIR, "overview"), exist_ok=True)
fig.savefig(os.path.join(FIGURE_DIR, "overview", "confounder_ppl_correlations.png"),
            dpi=150, bbox_inches="tight")
plt.show()

### Matching quality: confounder balance (before vs after matching)

In [11]:
plot_matched_sample_balance(df, settings["matched"], match_vars, FIGURE_DIR)
print("\n--- Per-diagnosis 1:k matched ---")
plot_matched_sample_balance(df, settings["matched_per_dx"], match_vars, FIGURE_DIR)

,Variable,HC (before),Patient (before),SMD (before),HC (after),Patient (after),SMD (after),p (after)
0,qwen_avg_words,20.30 ± 11.13,18.17 ± 9.65,0.204,18.87 ± 8.94,18.32 ± 9.26,0.061,0.5848
1,education,14.48 ± 3.15,13.14 ± 2.87,0.444,14.04 ± 2.77,13.77 ± 2.68,0.097,0.3833


Saved: figures/matched/matching_balance_love_plot.png

N before: 372 | N after matching: 324
SMD < 0.1 on all vars after matching: True

Match integrity:
  Max HC per patient-pair observed: 1
  Problematic match_id rows: 0

--- Per-diagnosis 1:k matched ---


,Variable,HC (before),Patient (before),SMD (before),HC (after),Patient (after),SMD (after),p (after)
0,qwen_avg_words,20.30 ± 11.13,18.17 ± 9.65,0.204,18.04 ± 8.66,18.20 ± 10.05,0.017,0.8631
1,education,14.48 ± 3.15,13.14 ± 2.87,0.444,13.67 ± 2.79,13.31 ± 2.93,0.125,0.2013


Saved: figures/matched/matching_balance_love_plot.png

N before: 372 | N after matching: 472
SMD < 0.1 on all vars after matching: False

Duplicate checks (within diagnosis):
  HC duplicates found: 0
  Patient duplicates found: 0

Match integrity:
  Max HC per patient-pair observed: 2
  Problematic match_id rows: 0

Per-diagnosis distance summary available in return dict:
  results['distance_summary_by_diagnosis']
  results['distance_cutoff_summary_by_diagnosis']


{'balance':          Variable    HC (before) Patient (before)  SMD (before)    HC (after)  \
 0  qwen_avg_words  20.30 ± 11.13     18.17 ± 9.65      0.204245  18.04 ± 8.66   
 1       education   14.48 ± 3.15     13.14 ± 2.87      0.444187  13.67 ± 2.79   
 
   Patient (after)  SMD (after)  p (after)  
 0   18.20 ± 10.05     0.017176   0.863088  
 1    13.31 ± 2.93     0.125483   0.201320  ,
 'dup_hc_within_diagnosis': Empty DataFrame
 Columns: [match_diagnosis, file_id, n]
 Index: [],
 'dup_patient_within_diagnosis': Empty DataFrame
 Columns: [match_diagnosis, file_id, n]
 Index: [],
 'match_integrity':      match_id  n_pat  n_hc  n_dx
 0           0      1     1     1
 1           1      1     1     1
 2           2      1     2     1
 3           3      1     2     1
 4           4      1     2     1
 ..        ...    ...   ...   ...
 155       155      1     2     1
 156       156      1     2     1
 157       157      1     2     1
 158       158      1     2     1
 159       159 

## 3. Loop: Group comparison & clinical correlations for all 3 settings

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu


def plot_raincloud_by_diagnosis(
    df: pd.DataFrame,
    ppl_var: str,
    diag_order: list[str] | None = None,
    diag_colors: dict[str, str] | None = None,
    title: str = "",
    ylabel: str = "Perplexity",
    save_path: str | None = None,
    figsize: tuple = (10, 5),
    bracket_pairs: list[tuple[str, str]] | None = None,
    test_func=mannwhitneyu,
    jitter: float = 0.15,
    point_size: float = 12,
    alpha_violin: float = 0.35,
    alpha_points: float = 0.5,
    dpi: int = 150,
):
    """Raincloud plot (half-violin + jittered strip + boxplot) per diagnosis
    with optional significance brackets.

    Parameters
    ----------
    df : DataFrame with columns ``ppl_var`` and ``"diagnosis"``.
    bracket_pairs : list of (diagA, diagB) tuples to test and annotate.
        Defaults to every diagnosis vs "HC".
    """
    from matplotlib.patches import FancyArrowPatch

    plot_df = df[["diagnosis", ppl_var]].dropna().copy()

    # Resolve ordering / colours
    if diag_order is None:
        diag_order = sorted(plot_df["diagnosis"].unique())
    present = [d for d in diag_order if d in plot_df["diagnosis"].unique()]
    plot_df["diagnosis"] = pd.Categorical(
        plot_df["diagnosis"], categories=present, ordered=True
    )

    if diag_colors is None:
        palette = sns.color_palette("Set2", len(present))
        diag_colors = dict(zip(present, palette))
    palette = {d: diag_colors[d] for d in present if d in diag_colors}

    # Default brackets: each diagnosis vs HC
    if bracket_pairs is None and "HC" in present:
        bracket_pairs = [("HC", d) for d in present if d != "HC"]

    fig, ax = plt.subplots(figsize=figsize)

    # ── Half-violin (right side only) ──
    vp = ax.violinplot(
        [plot_df.loc[plot_df["diagnosis"] == d, ppl_var].values for d in present],
        positions=range(len(present)),
        showextrema=False,
        widths=0.6,
    )
    for i, body in enumerate(vp["bodies"]):
        # Clip to right half
        m = np.mean(body.get_paths()[0].vertices[:, 0])
        body.get_paths()[0].vertices[:, 0] = np.clip(
            body.get_paths()[0].vertices[:, 0], m, None
        )
        body.set_facecolor(palette.get(present[i], "#999"))
        body.set_alpha(alpha_violin)
        body.set_edgecolor("none")

    # ── Box (narrow, left-shifted) ──
    bp = ax.boxplot(
        [plot_df.loc[plot_df["diagnosis"] == d, ppl_var].values for d in present],
        positions=range(len(present)),
        widths=0.12,
        patch_artist=True,
        showfliers=False,
        zorder=3,
    )
    for i, patch in enumerate(bp["boxes"]):
        patch.set_facecolor(palette.get(present[i], "#999"))
        patch.set_alpha(0.7)
    for element in ["whiskers", "caps", "medians"]:
        for line in bp[element]:
            line.set_color("black")
            line.set_linewidth(0.8)

    # ── Jittered strip (left side) ──
    for i, d in enumerate(present):
        vals = plot_df.loc[plot_df["diagnosis"] == d, ppl_var].values
        x = np.full_like(vals, i, dtype=float) - 0.2 + np.random.uniform(
            -jitter, 0, size=len(vals)
        )
        ax.scatter(
            x, vals,
            s=point_size,
            color=palette.get(d, "#999"),
            alpha=alpha_points,
            edgecolor="none",
            zorder=2,
        )

    # ── Significance brackets ──
    if bracket_pairs:
        y_max = plot_df[ppl_var].max()
        y_range = plot_df[ppl_var].max() - plot_df[ppl_var].min()
        step = y_range * 0.06
        y_start = y_max + y_range * 0.05

        for k, (a, b) in enumerate(bracket_pairs):
            if a not in present or b not in present:
                continue
            va = plot_df.loc[plot_df["diagnosis"] == a, ppl_var].values
            vb = plot_df.loc[plot_df["diagnosis"] == b, ppl_var].values
            if len(va) < 3 or len(vb) < 3:
                continue
            _, p = test_func(va, vb, alternative="two-sided")

            if p >= 0.05:
                label = "ns"
            elif p < 0.001:
                label = "***"
            elif p < 0.01:
                label = "**"
            else:
                label = "*"

            x1, x2 = present.index(a), present.index(b)
            y = y_start + k * step
            ax.plot([x1, x1, x2, x2], [y, y + step * 0.3, y + step * 0.3, y],
                    lw=0.8, color="black")
            ax.text(
                (x1 + x2) / 2, y + step * 0.35, label,
                ha="center", va="bottom", fontsize=8, color="black",
            )

    ax.set_xticks(range(len(present)))
    ax.set_xticklabels(present, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    if title:
        ax.set_title(title, fontsize=12)
    ax.grid(axis="y", alpha=0.2)
    sns.despine(ax=ax)
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    return fig

In [13]:
#how many dropped without_outliers compared to with outliers
n_with_outliers = len(df)
n_without_outliers = len(df.dropna(subset=[PPL_VAR]))

In [14]:
def plot_individual_clinical_vars(
    settings,
    setting_ppl_vars,
    merged_corr,
    MODEL,
    FIGURE_DIR,
    residualize_ppl,
    DIAG_ORDER,
    DIAG_COLORS,
    var_list,  # New: Pass your list of variables here
    setting_name="without_outliers",
):
    nice = {
        "GAF": "GAF",
        "TLI_Disorganization": "TLI Disorganization",
        "verbal_IQ": "Verbal IQ",
        "SAPS_PositiveFormaleDenkstoerung_ENIGMA": "SAPS Positive Formal Thought Disorder",
        "SAPS_Sum_ENIGMA": "SAPS Sum",
        "YMRS_Sum": "YMRS Sum",
        "SANS_AbulieApathie_ENIGMA": "SANS Abulia/Apathy",
        "SAPS_Wahn_ENIGMA": "SAPS Delusions",
        "SAPS_BizarresVerhalten_ENIGMA": "SAPS Bizarre Behavior",
        "LexicalVF": "Lexical Verbal Fluency",
    }

    def fmt_p(p):
        if p is None or (isinstance(p, float) and np.isnan(p)):
            return "NA"
        return f"{p:.4f}" if p >= 1e-4 else f"{p:.2e}"

    # ── Prepare data (same as original) ──────────────────────────
    wo_df = settings[setting_name].copy().reset_index(drop=True)
    wo_ppl = setting_ppl_vars[setting_name]
    wo_resid_col = f"{wo_ppl}_person_resid"

    wo_df["sex_num"] = wo_df["sex"].map({
        "M": 0, "F": 1, "m": 0, "f": 1,
        "male": 0, "female": 1, "Male": 0, "Female": 1,
    })

    confounders = ["sex_num", "age", "education", f"{MODEL}_avg_words"]
    wo_resid = residualize_ppl(wo_df, wo_ppl, confounders)
    wo_df[wo_resid_col] = np.nan
    wo_df.loc[wo_resid.index, wo_resid_col] = wo_resid

    wo_patients = wo_df[wo_df["is_patient"] == 1].copy()
    mc = merged_corr.set_index("Clinical Variable")

    # Filter var_list to only include variables present in the dataframe
    valid_targets = [c for c in var_list if c in wo_patients.columns]
    
    print(f"{'Variable':<40} | {'rho':>6} | {'p_FDR':>10} | {'p_int':>10}")
    print("-" * 75)

    for cvar in valid_targets:
        # Create a new figure for each variable
        fig, ax = plt.subplots(figsize=(5, 5))
        
        sub = wo_patients[[wo_resid_col, cvar, "diagnosis"]].dropna()
        if len(sub) < 5:
            plt.close(fig)
            continue

        # 1. Detailed: Scatter by diagnosis (No legend)
        for dx in [d for d in DIAG_ORDER if d != "HC" and d in sub["diagnosis"].values]:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(
                s[wo_resid_col], s[cvar],
                color=DIAG_COLORS.get(dx, "#888"),
                alpha=0.60, s=40,
                edgecolors="white", linewidths=0.3,
            )

        # 2. Not Detailed: Overall regression line
        if sub[wo_resid_col].nunique() > 1:
            z = np.polyfit(sub[wo_resid_col], sub[cvar], 1)
            xs = np.linspace(sub[wo_resid_col].min(), sub[wo_resid_col].max(), 100)
            ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.8, linewidth=2)

        # Formatting
        ax.set_xlabel("Deconfounded LM loss", fontsize=11)
        ax.set_ylabel(nice.get(cvar, cvar), fontsize=11)
        ax.grid(alpha=0.2)
        
        # Pull stats for printing
        rho_val = mc.loc[cvar, "r_resid"] if cvar in mc.index else np.nan
        p_fdr_val = mc.loc[cvar, "p_fdr_resid"] if cvar in mc.index else np.nan
        f_int, p_int = diagnosis_interaction_test(sub, wo_resid_col, cvar)

        # Print stats to console
        print(f"{cvar:<40} | {rho_val:>6.3f} | {fmt_p(p_fdr_val):>10} | {fmt_p(p_int):>10}")

        # Save individual plot
        out_dir = os.path.join(FIGURE_DIR, "individual_plots")
        os.makedirs(out_dir, exist_ok=True)
        save_path = os.path.join(out_dir, f"plot_{cvar}.png")
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show() # Shows them one by one in your notebook/IDE
        plt.close(fig)

    return

MAIN ANALYSES! HERE

In [15]:
import statsmodels.api as sm

# Map each setting to the PPL variable it should use
setting_ppl_vars = {
    #"with_outliers": PPL_VAR,
    "without_outliers": f"{PPL_VAR}",
    #"matched": PPL_VAR,
    #"matched_without_outliers": f"{PPL_VAR}",
    #"matched_per_dx": PPL_VAR,
    #"matched_per_dx_without_outliers": f"{PPL_VAR}",
}

all_results = {}
all_corr_dfs = {}

for setting_name, setting_df in settings.items():
    #check if setting_name is in setting_ppl_vars, if not, skip with a warning
    if setting_name not in setting_ppl_vars:
        print(f"Skipping: No PPL variable defined for setting '{setting_name}'. Skipping analyses for this setting.")
        continue
    ppl_col = setting_ppl_vars[setting_name]
    print(f"\n{'='*60}")
    print(f"Setting: {setting_name} | PPL variable: {ppl_col}")
    print(f"{'='*60}")

    # ── Standard analysis ─────────────────────────────────────────
    result = plot_setting(
        df=setting_df,
        ppl_var=ppl_col,
        setting_name=setting_name,
        figure_dir=FIGURE_DIR,
        clinical_vars=clinical_vars,
    )

    # ── Deconfounding analysis ────────────────────────────────────
    print(f"\n--- Deconfounding ({setting_name}) ---")
    sdf = setting_df.copy()
    sdf = sdf.reset_index(drop=True)
    sdf["log_total_words"] = np.log1p(sdf[WC_VAR])
    sdf["log_avg_words"] = np.log1p(sdf["qwen_avg_words"])
    confounders = [c if c != WC_VAR else "log_total_words" for c in CONFOUNDER_VARS]
    print(f"Confounders: {', '.join(confounders)}")
    ols_confounders = CONFOUNDER_VARS.copy()
    ols_confounders.append("qwen_avg_words")
    #if "log_total_words" not in ols_confounders:
        #ols_confounders.append("log_total_words")
    if "log_avg_words" not in ols_confounders:
        ols_confounders.append("log_avg_words")

    resid = residualize_ppl(sdf, ppl_col, ols_confounders, fit_on_all=True)
    resid_col = f"{ppl_col}_resid"
    sdf[resid_col] = np.nan
    sdf.loc[resid.index, resid_col] = resid

    ols_confounders_diag = ols_confounders + ["diagnosis"] 
    resid_diag = residualize_ppl(sdf, ppl_col, ols_confounders_diag, fit_on_all=True)
    resid_diag_col = f"{ppl_col}_resid_diag"
    sdf[resid_diag_col] = np.nan
    sdf.loc[resid_diag.index, resid_diag_col] = resid_diag

    # Group comparison: raw vs residualized
    comp_raw = group_comparison(sdf, ppl_col)
    comp_resid = group_comparison(sdf.dropna(subset=[resid_col]), resid_col)
    print(f"  Raw:          p(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    print(f"  Residualized: p(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")

    # Clinical correlations: raw vs residualized
    patients_s = sdf[sdf["is_patient"] == 1].copy()
    corr_raw = clinical_correlations(patients_s, ppl_col, clinical_vars_corr)
    corr_resid = clinical_correlations(
        patients_s.dropna(subset=[resid_col]), resid_col, clinical_vars_corr
    )
    corr_resid_diag = clinical_correlations(
        patients_s.dropna(subset=[resid_diag_col]), resid_diag_col, clinical_vars_corr
    )
    merged_corr = corr_raw.merge(
        corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
    )
    merged_corr = merged_corr.merge(
        corr_resid_diag, on="Clinical Variable"
    )
    rename_dict = {}
    for col in corr_resid_diag.columns:
        if col != "Clinical Variable":
            rename_dict[col] = f"{col}_resid_diag"
    merged_corr.rename(columns=rename_dict, inplace=True)

    print(f"\n  Running subgroup clinical correlations...")
    unique_diags = patients_s["diagnosis"].dropna().unique()
    
    for dx in unique_diags:
        dx_patients = patients_s[patients_s["diagnosis"] == dx].dropna(subset=[resid_col])
        
        # Only run correlation if there is a reasonable sample size (e.g., n >= 10)
        if len(dx_patients) >= 10:
            print(f"    -> {dx} (n={len(dx_patients)})")
            corr_dx = clinical_correlations(dx_patients, resid_col, clinical_vars_corr)
            
            # Rename columns to append the diagnosis name (e.g., r_resid_SSD)
            rename_dict_dx = {}
            for col in corr_dx.columns:
                if col != "Clinical Variable":
                    rename_dict_dx[col] = f"{col}_resid_{dx}"
            corr_dx.rename(columns=rename_dict_dx, inplace=True)
            
            # Merge this subgroup into the main table
            merged_corr = merged_corr.merge(corr_dx, on="Clinical Variable", how="left")
        else:
            print(f"    -> Skipping {dx}: n={len(dx_patients)} is too small for correlation.")
    # Format, clean, and save clinical correlation tables
    merged_corr = format_and_save_clinical_correlations(
        merged_corr, setting_name, FIGURE_DIR, fmt_p
    )

    # Pairwise vs HC: raw vs residualized
    from functions.analysis.statistics import pairwise_vs_hc, pairwise_vs_matched_hc, kruskal_wallis_test
    is_matched = "match_id" in sdf.columns
    pw_func = pairwise_vs_matched_hc if is_matched else pairwise_vs_hc
    pw_raw = pw_func(sdf, ppl_col)
    pw_resid = pw_func(sdf.dropna(subset=[resid_col]), resid_col)
    kw_raw_stat, kw_raw_p = kruskal_wallis_test(sdf, ppl_col)
    kw_res_stat, kw_res_p = kruskal_wallis_test(
        sdf.dropna(subset=[resid_col]), resid_col
    )
    print(f"\n  Kruskal-Wallis — Raw: H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
    print(f"  Kruskal-Wallis — Resid: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

    # ── Per-diagnosis table: each diagnosis vs HC ─────────────────
    pw_raw_df = pd.DataFrame(pw_raw)
    pw_res_df = pd.DataFrame(pw_resid)
    pw_merged = pw_raw_df[["Diagnosis", "n"]].copy()
    if "n(HC)" in pw_raw_df.columns:
        pw_merged.insert(1, "n(HC)", pw_raw_df["n(HC)"])
    else:
        pw_merged.insert(1, "n(HC)", len(sdf[sdf["diagnosis"] == "HC"]))
    pw_merged["d (raw)"] = pw_raw_df["Cohen's d"]
    pw_merged["p (raw)"] = pw_raw_df["p"]
    pw_merged["d (resid)"] = pw_res_df["Cohen's d"]
    pw_merged["p (resid)"] = pw_res_df["p"]
    pw_merged["sig (raw)"] = pw_merged["p (raw)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    pw_merged["sig (resid)"] = pw_merged["p (resid)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    print(f"\n  Per-diagnosis vs HC — {setting_name}:")
    display(pw_merged.style
        .format({"d (raw)": "{:.3f}", "p (raw)": "{:.4f}",
                 "d (resid)": "{:.3f}", "p (resid)": "{:.4f}"})
        .set_caption(f"Diagnosis vs HC — {setting_name}")
    )

    # ── Deconfounded visualisation ────────────────────────────────
    deconf_dir = os.path.join(FIGURE_DIR, setting_name, "deconfounded")
    confounded_dir = os.path.join(FIGURE_DIR, setting_name, "confounded")
    os.makedirs(deconf_dir, exist_ok=True)
    os.makedirs(confounded_dir, exist_ok=True)

    # Plot 1: raw vs residualized (uses plotting_service helper)
    plot_raw_vs_residualized(
        sdf, ppl_col, WC_VAR,
        confounders=CONFOUNDER_VARS,
        save_path=os.path.join(deconf_dir, "raw_vs_residualized.png"),
    )

    # Plot 1b: HC vs Patient boxplot after deconfounding
    plot_group_boxplot(
        sdf.dropna(subset=[resid_col]), resid_col,
        title=f"",
        save_path=os.path.join(deconf_dir, "group_boxplot_deconfounded_resid.png"),
        ylabel="Median PPL",
        plot_style="raincloud"
    )
    #raw plot
    plot_group_boxplot(
        sdf.dropna(subset=[resid_col]), ppl_col,
        title=f" ",
        save_path=os.path.join(confounded_dir, "group_boxplot_raincloud.png"),
        ylabel="Loss Median",
        plot_style="raincloud"
    )

    plot_raincloud_by_diagnosis(
        sdf.dropna(subset=[resid_col]),
        ppl_var=resid_col,
        diag_order=DIAG_ORDER,
        diag_colors=DIAG_COLORS,
        title=f"",
        ylabel="Mean Loss",
        save_path=os.path.join(deconf_dir, "raincloud_by_diagnosis_resid.png"),
    )

    plot_raincloud_by_diagnosis(
        sdf.dropna(subset=[ppl_col]),
        ppl_var=ppl_col,
        diag_order=DIAG_ORDER,
        diag_colors=DIAG_COLORS,
        title=f"",
        ylabel="Mean Loss",
        save_path=os.path.join(confounded_dir, "raincloud_by_diagnosis.png"),
    )

    # Plot 2: deconfounded comparison figure
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    from scipy.stats import spearmanr as _spearmanr

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Row 1 col 0: Raw PPL violin
    ax = axes[0, 0]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][ppl_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Raw {ppl_col}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

    # Row 1 col 1: Residualized PPL violin
    ax = axes[0, 1]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][resid_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
    ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

    # Row 1 col 2: Cohen's d comparison
    ax = axes[0, 2]
    ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
           color=["#3498DB", "#E74C3C"], alpha=0.8)
    ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
    ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

    # Row 2 col 0: Clinical correlations bar chart (plot subset only)
    ax = axes[1, 0]
    x = np.arange(len(clinical_vars))
    w = 0.35
    bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
    bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
    ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
    ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
    ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
    ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

    # Row 2 col 1: Scatter — residualized PPL vs TLI Impoverished
    ax = axes[1, 1]
    sub = patients_s[[resid_col, "TLI_Impoverished", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["TLI_Impoverished"])
        z = np.polyfit(sub[resid_col], sub["TLI_Impoverished"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
        ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    # Row 2 col 2: Scatter — residualized PPL vs GAF
    ax = axes[1, 2]
    sub = patients_s[[resid_col, "GAF", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["GAF"])
        z = np.polyfit(sub[resid_col], sub["GAF"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
        ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    fig.suptitle(f"Deconfounded analysis — {setting_name}", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(deconf_dir, "deconfounded_analysis.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Neutralitiy test confounders")
    neutrality_df = check_confounder_neutrality(sdf, ppl_col, resid_col, ols_confounders)
    plot_neutrality_check( sdf, ppl_col, resid_col, "log_avg_words", save_path=os.path.join(deconf_dir, "neutrality_check_log_words.png"))
    print(neutrality_df)
    # ── Store results ─────────────────────────────────────────────
    result["deconfounded"] = {
        "comparison_raw": comp_raw,
        "comparison_resid": comp_resid,
        "correlations_raw": corr_raw,
        "correlations_resid": corr_resid,
        "merged_correlations": merged_corr,
        "pairwise_raw": pw_raw,
        "pairwise_resid": pw_resid,
        "kruskal_raw": (kw_raw_stat, kw_raw_p),
        "kruskal_resid": (kw_res_stat, kw_res_p),
    }
    all_results[setting_name] = result
    all_corr_dfs[setting_name] = result["correlations"]

Skipping: No PPL variable defined for setting 'with_outliers'. Skipping analyses for this setting.

Setting: without_outliers | PPL variable: qwen_loss_median
Saved: figures/without_outliers/01_group_boxplot.png

without_outliers — qwen_loss_median:
  Variable: qwen_loss_median
  Control (M±SD): 4.98±0.54
  Patient (M±SD): 5.16±0.58
  Control Median: 4.947167678806932
  Patient Median: 5.167344473541624
  t: 3.2018213914099247
  p(t): 0.001484161725060241
  U: 20678.0
  p(MW): 0.0006937507408438543
  Cohens d: 0.3327326362633919
  r: 0.20381906037142694

Clinical correlations (without_outliers):
Clinical Variable         r        p   n    p_fdr  sig_fdr
              GAF -0.261681 0.000297 187 0.000297     True
 TLI_Impoverished  0.306397 0.000026 182 0.000052     True
Saved: figures/without_outliers/02_clinical_scatter.png
Saved: figures/without_outliers/03_diagnostic_groups.png
Saved: figures/without_outliers/04_extreme_groups.png
Saved: figures/without_outliers/05_tli_subscales.png


,Diagnosis,n(HC),n,d (raw),p (raw),d (resid),p (resid),sig (raw),sig (resid)
0,BD,178,26,0.517,0.0079,0.503,0.0607,**,ns
1,MDD,178,119,0.136,0.1268,0.010,0.6726,ns,ns
2,SSD,178,48,0.743,0.0000,0.478,0.0055,***,**


Saved: figures/without_outliers/deconfounded/raw_vs_residualized.png
Saved: figures/without_outliers/deconfounded/group_boxplot_deconfounded_resid.png
Saved: figures/without_outliers/confounded/group_boxplot_raincloud.png
Neutralitiy test confounders
Saved neutrality check plot: figures/without_outliers/deconfounded/neutrality_check_log_words.png
       Confounder   r (Raw)       p (Raw)  r (Resid)  p (Resid)
0         sex_num -0.118264  2.271262e-02  -0.017012   0.743972
1             age  0.187006  2.925199e-04   0.009628   0.853365
2  qwen_avg_words -0.784505  1.403148e-78   0.028208   0.588093
3       education -0.353130  2.456067e-12  -0.011924   0.818948
4  qwen_avg_words -0.784505  1.403148e-78   0.028208   0.588093
5   log_avg_words -0.784505  1.403148e-78   0.028208   0.588093
Skipping: No PPL variable defined for setting 'matched'. Skipping analyses for this setting.
Skipping: No PPL variable defined for setting 'matched_without_outliers'. Skipping analyses for this setting.


In [ ]:
import statsmodels.api as sm

# Map each setting to the PPL variable it should use
setting_ppl_vars = {
    #"with_outliers": PPL_VAR,
    "without_outliers": f"{PPL_VAR}",
    #"matched": PPL_VAR,
    #"matched_without_outliers": f"{PPL_VAR}",
    #"matched_per_dx": PPL_VAR,
    #"matched_per_dx_without_outliers": f"{PPL_VAR}",
}

all_results = {}
all_corr_dfs = {}

for setting_name, setting_df in settings.items():
    #check if setting_name is in setting_ppl_vars, if not, skip with a warning
    if setting_name not in setting_ppl_vars:
        print(f"Skipping: No PPL variable defined for setting '{setting_name}'. Skipping analyses for this setting.")
        continue
    ppl_col = setting_ppl_vars[setting_name]
    print(f"\n{'='*60}")
    print(f"Setting: {setting_name} | PPL variable: {ppl_col}")
    print(f"{'='*60}")

    # ── Standard analysis ─────────────────────────────────────────
    result = plot_setting(
        df=setting_df,
        ppl_var=ppl_col,
        setting_name=setting_name,
        figure_dir=FIGURE_DIR,
        clinical_vars=clinical_vars,
    )

    # ── Deconfounding analysis ────────────────────────────────────
    print(f"\n--- Deconfounding ({setting_name}) ---")
    sdf = setting_df.copy()
    sdf = sdf.reset_index(drop=True)
    sdf["log_total_words"] = np.log1p(sdf[WC_VAR])
    sdf["log_avg_words"] = np.log1p(sdf["qwen_avg_words"])
    confounders = [c if c != WC_VAR else "log_total_words" for c in CONFOUNDER_VARS]
    print(f"Confounders: {', '.join(confounders)}")
    ols_confounders = CONFOUNDER_VARS.copy()
    ols_confounders.append("qwen_avg_words")
    #if "log_total_words" not in ols_confounders:
        #ols_confounders.append("log_total_words")
    if "log_avg_words" not in ols_confounders:
        ols_confounders.append("log_avg_words")

    resid = residualize_ppl(sdf, ppl_col, ols_confounders, fit_on_all=True)
    resid_col = f"{ppl_col}_resid"
    sdf[resid_col] = np.nan
    sdf.loc[resid.index, resid_col] = resid

    ols_confounders_diag = ols_confounders + ["diagnosis"] 
    resid_diag = residualize_ppl(sdf, ppl_col, ols_confounders_diag, fit_on_all=True)
    resid_diag_col = f"{ppl_col}_resid_diag"
    sdf[resid_diag_col] = np.nan
    sdf.loc[resid_diag.index, resid_diag_col] = resid_diag

    # Group comparison: raw vs residualized
    comp_raw = group_comparison(sdf, ppl_col)
    comp_resid = group_comparison(sdf.dropna(subset=[resid_col]), resid_col)
    print(f"  Raw:          p(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    print(f"  Residualized: p(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")

    # Clinical correlations: raw vs residualized
    patients_s = sdf[sdf["is_patient"] == 1].copy()
    corr_raw = clinical_correlations(patients_s, ppl_col, clinical_vars_corr)
    corr_resid = clinical_correlations(
        patients_s.dropna(subset=[resid_col]), resid_col, clinical_vars_corr
    )
    corr_resid_diag = clinical_correlations(
        patients_s.dropna(subset=[resid_diag_col]), resid_diag_col, clinical_vars_corr
    )
    merged_corr = corr_raw.merge(
        corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
    )
    merged_corr = merged_corr.merge(
        corr_resid_diag, on="Clinical Variable"
    )
    rename_dict = {}
    for col in corr_resid_diag.columns:
        if col != "Clinical Variable":
            rename_dict[col] = f"{col}_resid_diag"
    merged_corr.rename(columns=rename_dict, inplace=True)

    print(f"\n  Running subgroup clinical correlations...")
    unique_diags = patients_s["diagnosis"].dropna().unique()
    
    for dx in unique_diags:
        dx_patients = patients_s[patients_s["diagnosis"] == dx].dropna(subset=[resid_col])
        
        # Only run correlation if there is a reasonable sample size (e.g., n >= 10)
        if len(dx_patients) >= 10:
            print(f"    -> {dx} (n={len(dx_patients)})")
            corr_dx = clinical_correlations(dx_patients, resid_col, clinical_vars_corr)
            
            # Rename columns to append the diagnosis name (e.g., r_resid_SSD)
            rename_dict_dx = {}
            for col in corr_dx.columns:
                if col != "Clinical Variable":
                    rename_dict_dx[col] = f"{col}_resid_{dx}"
            corr_dx.rename(columns=rename_dict_dx, inplace=True)
            
            # Merge this subgroup into the main table
            merged_corr = merged_corr.merge(corr_dx, on="Clinical Variable", how="left")
        else:
            print(f"    -> Skipping {dx}: n={len(dx_patients)} is too small for correlation.")
    # Format, clean, and save clinical correlation tables
    merged_corr = format_and_save_clinical_correlations(
        merged_corr, setting_name, FIGURE_DIR, fmt_p
    )

    # Pairwise vs HC: raw vs residualized
    from functions.analysis.statistics import pairwise_vs_hc, pairwise_vs_matched_hc, kruskal_wallis_test
    is_matched = "match_id" in sdf.columns
    pw_func = pairwise_vs_matched_hc if is_matched else pairwise_vs_hc
    pw_raw = pw_func(sdf, ppl_col)
    pw_resid = pw_func(sdf.dropna(subset=[resid_col]), resid_col)
    kw_raw_stat, kw_raw_p = kruskal_wallis_test(sdf, ppl_col)
    kw_res_stat, kw_res_p = kruskal_wallis_test(
        sdf.dropna(subset=[resid_col]), resid_col
    )
    print(f"\n  Kruskal-Wallis — Raw: H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
    print(f"  Kruskal-Wallis — Resid: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

    # ── Per-diagnosis table: each diagnosis vs HC ─────────────────
    pw_raw_df = pd.DataFrame(pw_raw)
    pw_res_df = pd.DataFrame(pw_resid)
    pw_merged = pw_raw_df[["Diagnosis", "n"]].copy()
    if "n(HC)" in pw_raw_df.columns:
        pw_merged.insert(1, "n(HC)", pw_raw_df["n(HC)"])
    else:
        pw_merged.insert(1, "n(HC)", len(sdf[sdf["diagnosis"] == "HC"]))
    pw_merged["d (raw)"] = pw_raw_df["Cohen's d"]
    pw_merged["p (raw)"] = pw_raw_df["p"]
    pw_merged["d (resid)"] = pw_res_df["Cohen's d"]
    pw_merged["p (resid)"] = pw_res_df["p"]
    pw_merged["sig (raw)"] = pw_merged["p (raw)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    pw_merged["sig (resid)"] = pw_merged["p (resid)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    print(f"\n  Per-diagnosis vs HC — {setting_name}:")
    display(pw_merged.style
        .format({"d (raw)": "{:.3f}", "p (raw)": "{:.4f}",
                 "d (resid)": "{:.3f}", "p (resid)": "{:.4f}"})
        .set_caption(f"Diagnosis vs HC — {setting_name}")
    )

    # ── Deconfounded visualisation ────────────────────────────────
    deconf_dir = os.path.join(FIGURE_DIR, setting_name, "deconfounded")
    confounded_dir = os.path.join(FIGURE_DIR, setting_name, "confounded")
    os.makedirs(deconf_dir, exist_ok=True)
    os.makedirs(confounded_dir, exist_ok=True)

    # Plot 1: raw vs residualized (uses plotting_service helper)
    plot_raw_vs_residualized(
        sdf, ppl_col, WC_VAR,
        confounders=CONFOUNDER_VARS,
        save_path=os.path.join(deconf_dir, "raw_vs_residualized.png"),
    )

    # Plot 1b: HC vs Patient boxplot after deconfounding
    plot_group_boxplot(
        sdf.dropna(subset=[resid_col]), resid_col,
        title=f"",
        save_path=os.path.join(deconf_dir, "group_boxplot_deconfounded_resid.png"),
        ylabel="Median PPL",
        plot_style="raincloud"
    )
    #raw plot
    plot_group_boxplot(
        sdf.dropna(subset=[resid_col]), ppl_col,
        title=f" ",
        save_path=os.path.join(confounded_dir, "group_boxplot_raincloud.png"),
        ylabel="Loss Median",
        plot_style="raincloud"
    )

    plot_raincloud_by_diagnosis(
        sdf.dropna(subset=[resid_col]),
        ppl_var=resid_col,
        diag_order=DIAG_ORDER,
        diag_colors=DIAG_COLORS,
        title=f"",
        ylabel="Mean Loss",
        save_path=os.path.join(deconf_dir, "raincloud_by_diagnosis_resid.png"),
    )

    plot_raincloud_by_diagnosis(
        sdf.dropna(subset=[ppl_col]),
        ppl_var=ppl_col,
        diag_order=DIAG_ORDER,
        diag_colors=DIAG_COLORS,
        title=f"",
        ylabel="Mean Loss",
        save_path=os.path.join(confounded_dir, "raincloud_by_diagnosis.png"),
    )

    # Plot 2: deconfounded comparison figure
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    from scipy.stats import spearmanr as _spearmanr

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Row 1 col 0: Raw PPL violin
    ax = axes[0, 0]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][ppl_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Raw {ppl_col}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

    # Row 1 col 1: Residualized PPL violin
    ax = axes[0, 1]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][resid_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
    ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

    # Row 1 col 2: Cohen's d comparison
    ax = axes[0, 2]
    ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
           color=["#3498DB", "#E74C3C"], alpha=0.8)
    ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
    ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

    # Row 2 col 0: Clinical correlations bar chart (plot subset only)
    ax = axes[1, 0]
    x = np.arange(len(clinical_vars))
    w = 0.35
    bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
    bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
    ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
    ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
    ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
    ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

    # Row 2 col 1: Scatter — residualized PPL vs TLI Impoverished
    ax = axes[1, 1]
    sub = patients_s[[resid_col, "TLI_Impoverished", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["TLI_Impoverished"])
        z = np.polyfit(sub[resid_col], sub["TLI_Impoverished"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
        ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    # Row 2 col 2: Scatter — residualized PPL vs GAF
    ax = axes[1, 2]
    sub = patients_s[[resid_col, "GAF", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["GAF"])
        z = np.polyfit(sub[resid_col], sub["GAF"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
        ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    fig.suptitle(f"Deconfounded analysis — {setting_name}", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(deconf_dir, "deconfounded_analysis.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Neutralitiy test confounders")
    neutrality_df = check_confounder_neutrality(sdf, ppl_col, resid_col, ols_confounders)
    plot_neutrality_check( sdf, ppl_col, resid_col, "log_avg_words", save_path=os.path.join(deconf_dir, "neutrality_check_log_words.png"))
    print(neutrality_df)
    # ── Store results ─────────────────────────────────────────────
    result["deconfounded"] = {
        "comparison_raw": comp_raw,
        "comparison_resid": comp_resid,
        "correlations_raw": corr_raw,
        "correlations_resid": corr_resid,
        "merged_correlations": merged_corr,
        "pairwise_raw": pw_raw,
        "pairwise_resid": pw_resid,
        "kruskal_raw": (kw_raw_stat, kw_raw_p),
        "kruskal_resid": (kw_res_stat, kw_res_p),
    }
    all_results[setting_name] = result
    all_corr_dfs[setting_name] = result["correlations"]

Skipping: No PPL variable defined for setting 'with_outliers'. Skipping analyses for this setting.

Setting: without_outliers | PPL variable: qwen_loss_median
Saved: figures/without_outliers/01_group_boxplot.png

without_outliers — qwen_loss_median:
  Variable: qwen_loss_median
  Control (M±SD): 4.98±0.54
  Patient (M±SD): 5.16±0.58
  Control Median: 4.947167678806932
  Patient Median: 5.167344473541624
  t: 3.2018213914099247
  p(t): 0.001484161725060241
  U: 20678.0
  p(MW): 0.0006937507408438543
  Cohens d: 0.3327326362633919
  r: 0.20381906037142694

Clinical correlations (without_outliers):
Clinical Variable         r        p   n    p_fdr  sig_fdr
              GAF -0.261681 0.000297 187 0.000297     True
 TLI_Impoverished  0.306397 0.000026 182 0.000052     True
Saved: figures/without_outliers/02_clinical_scatter.png
Saved: figures/without_outliers/03_diagnostic_groups.png
Saved: figures/without_outliers/04_extreme_groups.png
Saved: figures/without_outliers/05_tli_subscales.png


,Diagnosis,n(HC),n,d (raw),p (raw),d (resid),p (resid),sig (raw),sig (resid)
0,BD,178,26,0.517,0.0079,0.503,0.0607,**,ns
1,MDD,178,119,0.136,0.1268,0.010,0.6726,ns,ns
2,SSD,178,48,0.743,0.0000,0.478,0.0055,***,**


Saved: figures/without_outliers/deconfounded/raw_vs_residualized.png
Saved: figures/without_outliers/deconfounded/group_boxplot_deconfounded_resid.png
Saved: figures/without_outliers/confounded/group_boxplot_raincloud.png
Neutralitiy test confounders
Saved neutrality check plot: figures/without_outliers/deconfounded/neutrality_check_log_words.png
       Confounder   r (Raw)       p (Raw)  r (Resid)  p (Resid)
0         sex_num -0.118264  2.271262e-02  -0.017012   0.743972
1             age  0.187006  2.925199e-04   0.009628   0.853365
2  qwen_avg_words -0.784505  1.403148e-78   0.028208   0.588093
3       education -0.353130  2.456067e-12  -0.011924   0.818948
4  qwen_avg_words -0.784505  1.403148e-78   0.028208   0.588093
5   log_avg_words -0.784505  1.403148e-78   0.028208   0.588093
Skipping: No PPL variable defined for setting 'matched'. Skipping analyses for this setting.
Skipping: No PPL variable defined for setting 'matched_without_outliers'. Skipping analyses for this setting.


In [16]:
from functions.analysis.statistics import evaluate_incremental_value
from functions.plotting.plots import plot_incremental_value_roc
print("ALL Diagnosis")
results = evaluate_incremental_value(
    df=df, 
    target="is_patient", 
    length_metric=f"{MODEL}_avg_words", 
    ppl_metric=PPL_VAR,
    confounder_cols=CONFOUNDER_VARS)

print(f"--- Multi-Model Comparison (Predicting Patients) ---")
print(f"{'Model Name':<20} | {'AIC':>8} | {'CV-AUC':>8}")
print("-" * 45)
for name, m in results['models'].items():
    print(f"{name:<20} | {m['aic']:>8.1f} | {m['auc']:>8.3f}")

print(f"\n--- Statistical Tests for Incremental Value ---")
print(f"PPL adds value beyond Word Length?      p = {results['lrt_p_vs_length']:.4f}")
print(f"PPL adds value beyond All Confounders?  p = {results['lrt_p_vs_confounders']:.4f}")
# 3. Generate the visualization
plot_incremental_value_roc(
   y_true=results['y_true'],
   models_dict=results['models'],
   save_path=os.path.join(FIGURE_DIR, "advanced", "comprehensive_incremental_value_roc_all.png"))

ALL Diagnosis
--- Multi-Model Comparison (Predicting Patients) ---
Model Name           |      AIC |   CV-AUC
---------------------------------------------
PPL Only             |    507.5 |    0.588
Length Only          |    512.3 |    0.555
Length + PPL         |    509.5 |    0.582
Confounders Only     |    502.5 |    0.603
Confounders + PPL    |    501.3 |    0.618

--- Statistical Tests for Incremental Value ---
PPL adds value beyond Word Length?      p = 0.0292
PPL adds value beyond All Confounders?  p = 0.0740
Saved: figures/advanced/comprehensive_incremental_value_roc_all.png


<Figure size 840x720 with 1 Axes>

In [17]:
from functions.analysis.statistics import evaluate_incremental_value
from functions.plotting.plots import plot_incremental_value_roc
print(f"Subanalyses SSD vs. HC")
df_ssd = df[df["diagnosis"].isin(["HC", "SSD"])].copy()
results = evaluate_incremental_value(
    df=df_ssd, 
    target="is_patient", 
    length_metric=f"{MODEL}_avg_words", 
    ppl_metric=PPL_VAR,
    confounder_cols=CONFOUNDER_VARS)

print(f"--- Multi-Model Comparison (Predicting Patients) ---")
print(f"{'Model Name':<20} | {'AIC':>8} | {'CV-AUC':>8}")
print("-" * 45)
for name, m in results['models'].items():
    print(f"{name:<20} | {m['aic']:>8.1f} | {m['auc']:>8.3f}")

print(f"\n--- Statistical Tests for Incremental Value ---")
print(f"PPL adds value beyond Word Length?      p = {results['lrt_p_vs_length']:.4f}")
print(f"PPL adds value beyond All Confounders?  p = {results['lrt_p_vs_confounders']:.4f}")
# 3. Generate the visualization
plot_incremental_value_roc(
   y_true=results['y_true'],
   models_dict=results['models'],
   save_path=os.path.join(FIGURE_DIR, "advanced", "comprehensive_incremental_value_roc_ssd.png"))

Subanalyses SSD vs. HC
--- Multi-Model Comparison (Predicting Patients) ---
Model Name           |      AIC |   CV-AUC
---------------------------------------------
PPL Only             |    219.0 |    0.678
Length Only          |    225.9 |    0.637
Length + PPL         |    221.0 |    0.669
Confounders Only     |    208.2 |    0.737
Confounders + PPL    |    204.1 |    0.750

--- Statistical Tests for Incremental Value ---
PPL adds value beyond Word Length?      p = 0.0085
PPL adds value beyond All Confounders?  p = 0.0133
Saved: figures/advanced/comprehensive_incremental_value_roc_ssd.png


<Figure size 840x720 with 1 Axes>

In [18]:
from functions.analysis.statistics import evaluate_incremental_value
from functions.plotting.plots import plot_incremental_value_roc
print(f"Subanalyses BD vs. HC")
df_bd = df[df["diagnosis"].isin(["HC", "BD"])].copy()
results = evaluate_incremental_value(
    df=df_bd, 
    target="is_patient", 
    length_metric=f"{MODEL}_avg_words", 
    ppl_metric=PPL_VAR,
    confounder_cols=CONFOUNDER_VARS)

print(f"--- Multi-Model Comparison (Predicting Patients) ---")
print(f"{'Model Name':<20} | {'AIC':>8} | {'CV-AUC':>8}")
print("-" * 45)
for name, m in results['models'].items():
    print(f"{name:<20} | {m['aic']:>8.1f} | {m['auc']:>8.3f}")

print(f"\n--- Statistical Tests for Incremental Value ---")
print(f"PPL adds value beyond Word Length?      p = {results['lrt_p_vs_length']:.4f}")
print(f"PPL adds value beyond All Confounders?  p = {results['lrt_p_vs_confounders']:.4f}")
# 3. Generate the visualization
plot_incremental_value_roc(
   y_true=results['y_true'],
   models_dict=results['models'],
   save_path=os.path.join(FIGURE_DIR, "advanced", "comprehensive_incremental_value_roc_BD.png"))

Subanalyses BD vs. HC
--- Multi-Model Comparison (Predicting Patients) ---
Model Name           |      AIC |   CV-AUC
---------------------------------------------
PPL Only             |    153.9 |    0.626
Length Only          |    159.0 |    0.458
Length + PPL         |    153.6 |    0.601
Confounders Only     |    161.9 |    0.455
Confounders + PPL    |    157.7 |    0.556

--- Statistical Tests for Incremental Value ---
PPL adds value beyond Word Length?      p = 0.0066
PPL adds value beyond All Confounders?  p = 0.0130
Saved: figures/advanced/comprehensive_incremental_value_roc_BD.png


<Figure size 840x720 with 1 Axes>

In [19]:
from functions.analysis.statistics import evaluate_incremental_value
from functions.plotting.plots import plot_incremental_value_roc
print(f"Subanalyses MDD vs. HC")
df_mdd = df[df["diagnosis"].isin(["HC", "MDD"])].copy()
results = evaluate_incremental_value(
    df=df_mdd, 
    target="is_patient", 
    length_metric=f"{MODEL}_avg_words", 
    ppl_metric=PPL_VAR,
    confounder_cols=CONFOUNDER_VARS)

print(f"--- Multi-Model Comparison (Predicting Patients) ---")
print(f"{'Model Name':<20} | {'AIC':>8} | {'CV-AUC':>8}")
print("-" * 45)
for name, m in results['models'].items():
    print(f"{name:<20} | {m['aic']:>8.1f} | {m['auc']:>8.3f}")

print(f"\n--- Statistical Tests for Incremental Value ---")
print(f"PPL adds value beyond Word Length?      p = {results['lrt_p_vs_length']:.4f}")
print(f"PPL adds value beyond All Confounders?  p = {results['lrt_p_vs_confounders']:.4f}")
# 3. Generate the visualization
plot_incremental_value_roc(
   y_true=results['y_true'],
   models_dict=results['models'],
   save_path=os.path.join(FIGURE_DIR, "advanced", "comprehensive_incremental_value_roc.png"))

Subanalyses MDD vs. HC
--- Multi-Model Comparison (Predicting Patients) ---
Model Name           |      AIC |   CV-AUC
---------------------------------------------
PPL Only             |    402.6 |    0.506
Length Only          |    402.5 |    0.491
Length + PPL         |    404.4 |    0.483
Confounders Only     |    401.3 |    0.494
Confounders + PPL    |    403.3 |    0.490

--- Statistical Tests for Incremental Value ---
PPL adds value beyond Word Length?      p = 0.6992
PPL adds value beyond All Confounders?  p = 0.9106
Saved: figures/advanced/comprehensive_incremental_value_roc.png


<Figure size 840x720 with 1 Axes>

In [20]:
sdf[resid_col]

0      0.141237
1     -0.461299
2      0.321593
3      0.076736
4      0.101300
         ...   
367    0.315592
368    0.100460
369   -0.046743
370    0.000806
371   -0.524478
Name: qwen_loss_median_resid, Length: 372, dtype: float64

In [21]:
sorted_corr = corr_raw.sort_values("r", key=abs, ascending=False)[["Clinical Variable","n","r", "p", "p_fdr"]]
sorted_corr.to_csv(os.path.join(FIGURE_DIR, "with_outliers", "confounded", "sorted_clinical_correlations.csv"), index=False)

## 4. Cross-setting comparison

In [22]:
plot_cross_setting(all_corr_dfs, figure_dir=FIGURE_DIR)

Saved: figures/cross_setting/robustness_bar.png
Saved: figures/cross_setting/robustness_scatter.png


In [23]:
# Summary table of all comparisons (raw + deconfounded)
summary_rows = []
for sname, res in all_results.items():
    row_raw = {"Setting": sname, "Analysis": "Raw"}
    row_raw.update(res["comparison"])
    summary_rows.append(row_raw)
    if "deconfounded" in res:
        row_resid = {"Setting": sname, "Analysis": "Residualized"}
        row_resid.update(res["deconfounded"]["comparison_resid"])
        summary_rows.append(row_resid)
summary_df = pd.DataFrame(summary_rows)
summary_df

,Setting,Analysis,Variable,Control (M±SD),Patient (M±SD),Control Median,Patient Median,t,p(t),U,p(MW),Cohens d,r
0,without_outliers,Raw,qwen_loss_median,4.98±0.54,5.16±0.58,4.947168,5.167344,3.201821,0.001484,20678.0,0.000694,0.332733,0.203819
1,without_outliers,Residualized,qwen_loss_median_resid,-0.03±0.33,0.03±0.36,-0.043145,0.058666,1.810864,0.070975,19129.0,0.058620,0.188185,0.113640


## 5. Alternative metrics comparison

In [24]:
alt_metrics = [
    f"{MODEL}_ppl_median", f"{MODEL}_ppl_mean",
    f"{MODEL}_ppl_p90", f"{MODEL}_ppl_p75",
    f"{MODEL}_ppl_std", f"{MODEL}_ppl_cv",
    f"{MODEL}_prop_high_ppl",
]

# Group comparison for each metric
alt_comp = pd.DataFrame([group_comparison(df, m) for m in alt_metrics if m in df.columns])
alt_comp

,Variable,Control (M±SD),Patient (M±SD),Control Median,Patient Median,t,p(t),U,p(MW),Cohens d,r
0,qwen_ppl_median,171.15±125.54,206.80±131.31,140.776986,175.420110,2.671476,0.007886,20786.0,0.000681,0.277276,0.203869
1,qwen_ppl_mean,45097.73±225987.97,58214.30±471306.31,1878.625703,1704.472117,0.337390,0.736014,17193.0,0.944211,0.035018,-0.004228
2,qwen_ppl_p90,3370.52±6420.79,5494.17±33035.65,1619.849624,1785.216327,0.843077,0.399730,18225.0,0.354871,0.087504,0.055543
3,qwen_ppl_p75,596.99±678.91,705.42±721.01,374.897976,480.612225,1.489837,0.137119,19859.0,0.012338,0.154632,0.150180
4,qwen_ppl_std,332445.33±1849779.54,328611.64±2247343.57,7312.675829,5988.889578,-0.017872,0.985751,16913.0,0.733676,-0.001855,-0.020445
5,qwen_ppl_cv,4.22±1.96,4.13±1.99,3.914802,3.801248,-0.457652,0.647471,16503.0,0.461744,-0.047500,-0.044191
6,qwen_prop_high_ppl,0.09±0.07,0.10±0.07,0.076505,0.081220,0.881587,0.378573,18313.0,0.312359,0.091501,0.060639


In [25]:
# Correlation heatmap: metrics × clinical variables
patients = df[df['is_patient'] == 1].copy()
r_matrix = pd.DataFrame(index=[m for m in alt_metrics if m in df.columns], columns=clinical_vars, dtype=float)
p_matrix = r_matrix.copy()

for metric in r_matrix.index:
    for cvar in clinical_vars:
        sub = patients[[metric, cvar]].dropna()
        if len(sub) >= 5:
            r, p = spearmanr(sub[metric], sub[cvar])
            r_matrix.loc[metric, cvar] = r
            p_matrix.loc[metric, cvar] = p

plot_metrics_heatmap(
    r_matrix.astype(float),
    p_matrix.astype(float),
    title="Alternative metrics × Clinical variables (Spearman r)",
    save_path=os.path.join(FIGURE_DIR, "advanced", "metrics_heatmap.png"),
)

Saved: figures/advanced/metrics_heatmap.png


<Figure size 1200x720 with 2 Axes>

## 6. TLI subscale analysis (Steiger's Z)

In [26]:
tli_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
#if tli in name
tli_vars = [col for col in clinical_vars_corr if "TLI" in col]

for metric in tli_metrics:
    sub = patients[[metric] + tli_vars].dropna()
    if len(sub) < 10:
        continue
    r_imp, _ = spearmanr(sub[metric], sub["TLI_Impoverished"])
    r_dis, _ = spearmanr(sub[metric], sub["TLI_Disorganization"])
    r_id, _ = spearmanr(sub["TLI_Impoverished"], sub["TLI_Disorganization"])
    z, p = steigers_z(r_imp, r_dis, r_id, len(sub))
    print(f"{metric}: r_imp={r_imp:.3f}, r_dis={r_dis:.3f}, Steiger Z={z:.3f}, p={p:.4f}")

qwen_loss_median: r_imp=0.306, r_dis=0.033, Steiger Z=2.513, p=0.0120
qwen_ppl_p90: r_imp=0.302, r_dis=-0.014, Steiger Z=2.893, p=0.0038
qwen_prop_high_ppl: r_imp=0.281, r_dis=-0.014, Steiger Z=2.684, p=0.0073


## 7. ROC analysis

In [27]:
# Define clinical impairment targets
patients = df[df['is_patient'] == 1].copy()
patients['GAF_impaired'] = (patients['GAF'] < 60).astype(int)
patients['TLI_dis_any'] = (patients['TLI_Disorganization'] > 0).astype(int)
patients['TLI_imp_any'] = (patients['TLI_Impoverished'] > 0).astype(int)

ppl_roc_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
roc_targets = {
    'GAF < 60': 'GAF_impaired',
    'TLI Disorganization > 0': 'TLI_dis_any',
    'TLI Impoverished > 0': 'TLI_imp_any',
}

all_rocs = {}
for target_label, target_col in roc_targets.items():
    rocs = compute_roc(patients, ppl_roc_metrics, target_col)
    all_rocs[target_label] = rocs
    for r in rocs:
        print(f"  {target_label} | {r['metric']}: AUC={r['auc']:.3f}")

plot_roc_curves(
    [],  # not used directly
    all_rocs,
    save_path=os.path.join(FIGURE_DIR, "advanced", "roc_curves.png"),
)

  GAF < 60 | qwen_loss_median: AUC=0.589
  GAF < 60 | qwen_ppl_p90: AUC=0.520
  GAF < 60 | qwen_prop_high_ppl: AUC=0.534
  TLI Disorganization > 0 | qwen_loss_median: AUC=0.586
  TLI Disorganization > 0 | qwen_ppl_p90: AUC=0.548
  TLI Disorganization > 0 | qwen_prop_high_ppl: AUC=0.547
  TLI Impoverished > 0 | qwen_loss_median: AUC=0.660
  TLI Impoverished > 0 | qwen_ppl_p90: AUC=0.674
  TLI Impoverished > 0 | qwen_prop_high_ppl: AUC=0.658
Saved: figures/advanced/roc_curves.png


<Figure size 2160x600 with 3 Axes>

In [28]:
# Combined metric: logistic regression CV
features_combined = [PPL_VAR, f"{MODEL}_ppl_std", f"{MODEL}_prop_high_ppl"]
for target_label, target_col in roc_targets.items():
    auc_combined = logistic_cv_auc(patients, features_combined, target_col)
    auc_singles = {m: logistic_cv_auc(patients, [m], target_col) for m in features_combined}
    print(f"{target_label}: combined AUC={auc_combined:.3f}")
    for m, a in auc_singles.items():
        print(f"  single {m}: AUC={a:.3f}")

GAF < 60: combined AUC=0.476
  single qwen_loss_median: AUC=0.570
  single qwen_ppl_std: AUC=0.573
  single qwen_prop_high_ppl: AUC=0.544


TLI Disorganization > 0: combined AUC=0.568
  single qwen_loss_median: AUC=0.568
  single qwen_ppl_std: AUC=0.555
  single qwen_prop_high_ppl: AUC=0.545
TLI Impoverished > 0: combined AUC=0.495
  single qwen_loss_median: AUC=0.655
  single qwen_ppl_std: AUC=0.546
  single qwen_prop_high_ppl: AUC=0.649


## 8. Confounder analysis

In [29]:
# Confounder visualisation
plot_advanced(
    df=df,
    patients=patients,
    ppl_var=PPL_VAR,
    wc_var=WC_VAR,
    figure_dir=FIGURE_DIR,
)

Saved: figures/advanced/confounder_analysis.png
Saved: figures/advanced/raw_vs_residualized.png
Saved: figures/advanced/raw_perplexity_basic.png


### 8b. Deconfounding: residualize PPL, then re-run group comparisons & clinical correlations

Instead of nested ANOVA (where confounders absorb the group signal), we regress out
confounders (log word count + education) from perplexity and then test the **residualized**
perplexity for group differences and clinical associations.

In [30]:
# ── Residualize perplexity ────────────────────────────────────────
# Regress out log(word_count) + education from PPL, keep the residuals.
import statsmodels.api as sm

df_deconf = df.copy()
df_deconf["log_words"] = np.log1p(df_deconf[WC_VAR])

confounders = CONFOUNDER_VARS

df_deconf[f"{PPL_VAR}_resid"] = np.nan
resid = residualize_ppl(df_deconf, PPL_VAR, confounders)
df_deconf.loc[resid.index, f"{PPL_VAR}_resid"] = resid

RESID_VAR = f"{PPL_VAR}_resid"
print(f"Confounders removed: {confounders}")
print(f"Residualized variable: {RESID_VAR}")
print(f"N with residuals: {df_deconf[RESID_VAR].notna().sum()}")

# ── Group comparison: raw vs residualized ────────────────────────
print("\n" + "=" * 60)
print("GROUP COMPARISON — Raw PPL")
print("=" * 60)
comp_raw = group_comparison(df_deconf, PPL_VAR)
for k, v in comp_raw.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("GROUP COMPARISON — Residualized PPL")
print("=" * 60)
comp_resid = group_comparison(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
for k, v in comp_resid.items():
    print(f"  {k}: {v}")

# Summary table
comp_summary = pd.DataFrame([
    {"Metric": "Raw PPL", **{k: v for k, v in comp_raw.items() if k != "Variable"}},
    {"Metric": "Residualized PPL", **{k: v for k, v in comp_resid.items() if k != "Variable"}},
])
comp_summary

Confounders removed: ['sex_num', 'age', 'qwen_avg_words', 'education']
Residualized variable: qwen_loss_median_resid
N with residuals: 371

GROUP COMPARISON — Raw PPL
  Variable: qwen_loss_median
  Control (M±SD): 4.98±0.54
  Patient (M±SD): 5.16±0.58
  Control Median: 4.947167678806932
  Patient Median: 5.167344473541624
  t: 3.2018213914099247
  p(t): 0.001484161725060241
  U: 20678.0
  p(MW): 0.0006937507408438543
  Cohens d: 0.3327326362633919
  r: 0.20381906037142694

GROUP COMPARISON — Residualized PPL
  Variable: qwen_loss_median_resid
  Control (M±SD): -0.00±0.39
  Patient (M±SD): 0.09±0.44
  Control Median: -0.027054985844519486
  Patient Median: 0.0737900198566761
  t: 2.1189062339216025
  p(t): 0.03476597945603341
  U: 19558.0
  p(MW): 0.02106930804117394
  Cohens d: 0.22019631048101962
  r: 0.13861559061535766


,Metric,Control (M±SD),Patient (M±SD),Control Median,Patient Median,t,p(t),U,p(MW),Cohens d,r
0,Raw PPL,4.98±0.54,5.16±0.58,4.947168,5.167344,3.201821,0.001484,20678.0,0.000694,0.332733,0.203819
1,Residualized PPL,-0.00±0.39,0.09±0.44,-0.027055,0.073790,2.118906,0.034766,19558.0,0.021069,0.220196,0.138616


In [31]:
from scipy.stats import spearmanr

# For each confounder, check:
# (a) confounder ↔ PPL
# (b) confounder ↔ clinical variable
# If (a) and (b) have SAME sign → suppressor (partialling out strengthens the clinical r)
# If (a) and (b) have OPPOSITE sign → classical confound (partialling out weakens it)

confounders_to_check = ["age", "education", f"{MODEL}_avg_words", "sex_num"]
clinical_to_check = ["YMRS_Sum", "SAPS_Sum_ENIGMA", "GAF", "verbal_IQ"]

patients = df_deconf[df_deconf["is_patient"] == 1].copy()
patients["sex_num"] = patients["sex"].map(
    {"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1}
)

rows = []
for conf in confounders_to_check:
    # confounder ↔ PPL
    sub = patients[[conf, PPL_VAR]].dropna()
    r_ppl, p_ppl = spearmanr(sub[conf], sub[PPL_VAR])

    for cvar in clinical_to_check:
        sub2 = patients[[conf, cvar]].dropna()
        if len(sub2) < 10:
            continue
        r_clin, p_clin = spearmanr(sub2[conf], sub2[cvar])

        same_sign = (r_ppl > 0) == (r_clin > 0)
        role = "suppressor" if same_sign else "classical confound"

        rows.append({
            "confounder": conf,
            "clinical_var": cvar,
            "r(conf↔PPL)": round(r_ppl, 3),
            "r(conf↔clin)": round(r_clin, 3),
            "same_sign": same_sign,
            "role": role,
        })

path_df = pd.DataFrame(rows)
print(path_df.to_string(index=False))

    confounder    clinical_var  r(conf↔PPL)  r(conf↔clin)  same_sign               role
           age        YMRS_Sum        0.224        -0.017      False classical confound
           age SAPS_Sum_ENIGMA        0.224         0.040       True         suppressor
           age             GAF        0.224         0.075       True         suppressor
           age       verbal_IQ        0.224         0.265       True         suppressor
     education        YMRS_Sum       -0.300        -0.168       True         suppressor
     education SAPS_Sum_ENIGMA       -0.300        -0.076       True         suppressor
     education             GAF       -0.300         0.215      False classical confound
     education       verbal_IQ       -0.300         0.300      False classical confound
qwen_avg_words        YMRS_Sum       -0.792        -0.297       True         suppressor
qwen_avg_words SAPS_Sum_ENIGMA       -0.792        -0.229       True         suppressor
qwen_avg_words             GAF  

In [32]:
# ── Clinical correlations: raw vs residualized ───────────────────
patients_deconf = df_deconf[df_deconf["is_patient"] == 1].copy()

print("CLINICAL CORRELATIONS — Raw PPL (patients only)")
corr_raw = clinical_correlations(patients_deconf, PPL_VAR, clinical_vars_corr)
print(corr_raw.to_string(index=False))

print("\nCLINICAL CORRELATIONS — Residualized PPL (patients only)")
corr_resid = clinical_correlations(
    patients_deconf.dropna(subset=[RESID_VAR]), RESID_VAR, clinical_vars_corr
)
print(corr_resid.to_string(index=False))

# Side-by-side comparison
merged_corr = corr_raw.merge(
    corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
)
merged_corr

CLINICAL CORRELATIONS — Raw PPL (patients only)
                      Clinical Variable         r            p   n    p_fdr  sig_fdr
                                    GAF -0.261681 2.972541e-04 187 0.000764     True
                       TLI_Impoverished  0.306397 2.591569e-05 182 0.000155     True
                    TLI_Disorganization  0.032612 6.620830e-01 182 0.662083    False
                              verbal_IQ -0.311850 1.552328e-05 185 0.000140     True
                       TLI_Non_specific  0.106315 1.531678e-01 182 0.172314    False
            SANS_AlogieParalogie_ENIGMA  0.191169 8.412345e-03 189 0.011648     True
SAPS_PositiveFormaleDenkstoerung_ENIGMA  0.256260 3.719351e-04 189 0.000837     True
                        SAPS_Sum_ENIGMA  0.293333 4.614012e-05 187 0.000208     True
                               YMRS_Sum  0.361632 3.175265e-07 189 0.000006     True
                             HAMD_Sum17  0.129854 7.571374e-02 188 0.090856    False
                 

,Clinical Variable,r_raw,p_raw,n_raw,p_fdr_raw,sig_fdr_raw,r_resid,p_resid,n_resid,p_fdr_resid,sig_fdr_resid
0,GAF,-0.261681,2.972541e-04,187,0.000764,True,-0.158121,0.030665,187,0.050179,False
1,TLI_Impoverished,0.306397,2.591569e-05,182,0.000155,True,0.243919,0.000906,182,0.004076,True
2,TLI_Disorganization,0.032612,6.620830e-01,182,0.662083,False,0.092475,0.214375,182,0.231890,False
3,verbal_IQ,-0.311850,1.552328e-05,185,0.000140,True,-0.302533,0.000028,185,0.000512,True
4,TLI_Non_specific,0.106315,1.531678e-01,182,0.172314,False,0.091552,0.219008,182,0.231890,False
5,SANS_AlogieParalogie_ENIGMA,0.191169,8.412345e-03,189,0.011648,True,0.128508,0.078020,189,0.108028,False
6,SAPS_PositiveFormaleDenkstoerung_ENIGMA,0.256260,3.719351e-04,189,0.000837,True,0.198329,0.006225,189,0.016006,True
7,SAPS_Sum_ENIGMA,0.293333,4.614012e-05,187,0.000208,True,0.235693,0.001165,187,0.004194,True
8,YMRS_Sum,0.361632,3.175265e-07,189,0.000006,True,0.264044,0.000241,189,0.002172,True
9,HAMD_Sum17,0.129854,7.571374e-02,188,0.090856,False,0.083822,0.252763,188,0.252763,False


In [33]:
import pandas as pd

def format_correlation_table(
    df,
    corr_symbol="ρ",       # Spearman = "ρ", Pearson = "r"
    corr_decimals=3,
    p_decimals=3,
    n_col="n_raw",
    stars_based_on="fdr"   # "fdr" or "raw"
):
    """
    Reformat Spearman/Pearson correlation results table for manuscript output.

    Stars:
    *   p < .05
    **  p < .01
    *** p < .001

    By default, stars are based on FDR-adjusted p-values.
    """

    out = df.copy()

    def format_corr(x):
        if pd.isna(x):
            return ""
        return f"{x:.{corr_decimals}f}"

    def format_p(x):
        if pd.isna(x):
            return ""
        threshold = 10 ** (-p_decimals)
        if x < threshold:
            return f"<{threshold:.{p_decimals}f}"
        return f"{x:.{p_decimals}f}"

    def p_to_stars(p):
        if pd.isna(p):
            return ""
        elif p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    if stars_based_on == "fdr":
        raw_sig_source = out["p_fdr_raw"]
        resid_sig_source = out["p_fdr_resid"]
    elif stars_based_on == "raw":
        raw_sig_source = out["p_raw"]
        resid_sig_source = out["p_resid"]
    else:
        raise ValueError("stars_based_on must be either 'fdr' or 'raw'.")

    formatted = pd.DataFrame({
        "Variable": out["Clinical Variable"],

        f"{corr_symbol} (raw)": out["r_raw"].apply(format_corr),
        "P (raw)": out["p_raw"].apply(format_p),
        "P FDR (raw)": out["p_fdr_raw"].apply(format_p),
        "Sig. raw": raw_sig_source.apply(p_to_stars),

        f"{corr_symbol} (res.)": out["r_resid"].apply(format_corr),
        "P (res.)": out["p_resid"].apply(format_p),
        "P FDR (res.)": out["p_fdr_resid"].apply(format_p),
        "Sig. res.": resid_sig_source.apply(p_to_stars),

        "n": out[n_col]
    })

    return formatted

In [34]:
sorted_clinical_corr_table = format_correlation_table(
    merged_corr.sort_values("p_fdr_raw", ascending=True),
    corr_symbol="ρ",      # use "r" if Pearson
    corr_decimals=3,
    p_decimals=3,
)



In [35]:
sorted_clinical_corr_table.to_csv(os.path.join(TABLE_DIR, "formatted_clinical_correlations.csv"), index=False)

In [36]:
# ── Visualisation: raw vs residualized ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Raw PPL
ax = axes[0, 0]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][PPL_VAR].dropna()
    ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Raw {PPL_VAR}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

# Row 1: Residualized PPL
ax = axes[0, 1]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][RESID_VAR].dropna()
    parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(clr); pc.set_alpha(0.6)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

# Row 1: Cohen's d comparison
ax = axes[0, 2]
ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
       color=["#3498DB", "#E74C3C"], alpha=0.8)
ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

# Row 2: Clinical correlations comparison
ax = axes[1, 0]
x = np.arange(len(clinical_vars))
w = 0.35
bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

# Row 2: Scatter — residualized PPL vs TLI Impoverished
ax = axes[1, 1]
sub = patients_deconf[[RESID_VAR, "TLI_Impoverished", "diagnosis"]].dropna()
if len(sub) >= 5:
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["TLI_Impoverished"])
    z = np.polyfit(sub[RESID_VAR], sub["TLI_Impoverished"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
    ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

# Row 2: Scatter — residualized PPL vs GAF
ax = axes[1, 2]
sub = patients_deconf[[RESID_VAR, "GAF", "diagnosis"]].dropna()
if len(sub) >= 5:
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["GAF"])
    z = np.polyfit(sub[RESID_VAR], sub["GAF"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
    ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

fig.suptitle("Deconfounded analysis: raw vs residualized perplexity", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "advanced", "deconfounded_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [37]:
# ── Per-diagnosis deconfounded comparison vs HC ──────────────────
from functions.analysis.statistics import pairwise_vs_hc, kruskal_wallis_test

print("PAIRWISE vs HC — Raw PPL")
pw_raw = pairwise_vs_hc(df_deconf, PPL_VAR)
print(pd.DataFrame(pw_raw).to_string(index=False))

print("\nPAIRWISE vs HC — Residualized PPL")
pw_resid = pairwise_vs_hc(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
print(pd.DataFrame(pw_resid).to_string(index=False))

print("\nKruskal-Wallis across diagnoses:")
kw_raw_stat, kw_raw_p = kruskal_wallis_test(df_deconf, PPL_VAR)
kw_res_stat, kw_res_p = kruskal_wallis_test(
    df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR
)
print(f"  Raw:          H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
print(f"  Residualized: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

PAIRWISE vs HC — Raw PPL
Diagnosis       U        p  Cohen's d   n
       BD  3062.0 0.007850   0.516970  26
      MDD 11699.0 0.126758   0.135928 119
      SSD  5917.0 0.000043   0.742532  48

PAIRWISE vs HC — Residualized PPL
Diagnosis       U        p  Cohen's d   n
       BD  3030.0 0.010939   0.572175  26
      MDD 11004.0 0.569524   0.007956 119
      SSD  5524.0 0.001852   0.578333  48

Kruskal-Wallis across diagnoses:
  Raw:          H=21.296, p=0.0001
  Residualized: H=14.937, p=0.0019


## Figure 2: Residualized PPL vs YMRS & SAPS (without_outliers)

In [38]:
wo_df = settings["with_outliers"].copy().reset_index(drop=True)

In [39]:
from scipy.stats import spearmanr as _spearmanr
from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
import math
import os
import numpy as np
import matplotlib.pyplot as plt

# Reconstruct the person-level deconfounded metric for without_outliers
wo_df = settings["with_outliers"].copy().reset_index(drop=True)
wo_ppl = setting_ppl_vars["with_outliers"]
wo_resid_col = f"{wo_ppl}_person_resid"

wo_df["sex_num"] = wo_df["sex"].map({
    "M": 0, "F": 1,
    "m": 0, "f": 1,
    "male": 0, "female": 1,
    "Male": 0, "Female": 1,
})

# Use numeric confounders only
confounders = [
    "sex_num",
    "age",
    "education",
    f"{MODEL}_avg_words",
]

wo_resid = residualize_ppl(wo_df, wo_ppl, confounders)

wo_df[wo_resid_col] = np.nan
wo_df.loc[wo_resid.index, wo_resid_col] = wo_resid

wo_patients = wo_df[wo_df["is_patient"] == 1].copy()

# Variables significant after FDR in the residualized/deconfounded analysis
clinical_targets = (
    merged_corr.loc[merged_corr["sig_fdr_resid"] == True, "Clinical Variable"]
    .dropna()
    .tolist()
)

# Keep only variables actually available in wo_patients
clinical_targets = [c for c in clinical_targets if c in wo_patients.columns]

print("FDR-significant residualized clinical targets:")
print(clinical_targets)

nice = {
    "GAF": "GAF",
    "TLI_Disorganization": "TLI Disorganization",
    "verbal_IQ": "Verbal IQ",
    "SAPS_PositiveFormaleDenkstoerung_ENIGMA": "SAPS formal thought disorder",
    "SAPS_Sum_ENIGMA": "SAPS Sum",
    "YMRS_Sum": "YMRS Sum",
    "SANS_AbulieApathie_ENIGMA": "SANS abulia/apathy",
    "SAPS_Wahn_ENIGMA": "SAPS delusions",
    "SAPS_BizarresVerhalten_ENIGMA": "SAPS bizarre behavior",
    "LexicalVF": "Lexical verbal fluency",
}

n = len(clinical_targets)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(6.2 * ncols, 4.8 * nrows),
    squeeze=False,
)

for i, cvar in enumerate(clinical_targets):
    ax = axes[i // ncols][i % ncols]

    sub = wo_patients[[wo_resid_col, cvar, "diagnosis"]].dropna()

    if len(sub) < 5:
        ax.text(
            0.5, 0.5,
            f"Not enough data for {nice.get(cvar, cvar)}",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
        ax.axis("off")
        continue

    # Scatter by diagnosis
    for dx in [d for d in DIAG_ORDER if d != "HC" and d in sub["diagnosis"].values]:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(
            s[wo_resid_col],
            s[cvar],
            color=DIAG_COLORS.get(dx, "#888"),
            label=dx,
            alpha=0.65,
            s=35,
            edgecolors="white",
            linewidths=0.3,
        )

    # Regression line + Spearman stats
    r, p = _spearmanr(sub[wo_resid_col], sub[cvar])

    if sub[wo_resid_col].nunique() > 1:
        z = np.polyfit(sub[wo_resid_col], sub[cvar], 1)
        xs = np.linspace(sub[wo_resid_col].min(), sub[wo_resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6, linewidth=1.2)

    p_str = f"p={p:.4f}" if p >= 0.0001 else f"p={p:.2e}"

    ax.set_xlabel("Person-level deconfounded length-adjusted LM loss", fontsize=10)
    ax.set_ylabel(nice.get(cvar, cvar), fontsize=10)
    ax.set_title(
        f"{nice.get(cvar, cvar)}\nρ={r:.3f}, {p_str}",
        fontsize=11,
    )
    ax.grid(alpha=0.2)

    if i == 0:
        ax.legend(fontsize=8, framealpha=0.8)

# Turn off empty panels
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")

fig.suptitle(
    "FDR-significant clinical associations with deconfounded length-adjusted LM loss",
    fontsize=14,
    y=1.002,
)

fig.tight_layout()

save_path = os.path.join(
    FIGURE_DIR,
    "main_figures",
    "fdr_significant_clinical_scatter_grid.png",
)

os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved to {save_path}")
plt.show()

KeyError: 'with_outliers'

In [ ]:
def plot_fdr_significant_clinical_scatter_grid_with_dx_lines(
    settings,
    setting_ppl_vars,
    merged_corr,
    MODEL,
    FIGURE_DIR,
    residualize_ppl,
    DIAG_ORDER,
    DIAG_COLORS,
    setting_name="without_outliers",
    min_points_panel=8,
    min_points_line=4,
    min_points_positive=8,
    save_name="fdr_significant_clinical_scatter_grid_hurdle.png",
    csv_prefix="fdr_significant_clinical_hurdle",
    include_diag_in_caption=True,
):
    import os
    import math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    from scipy.stats import spearmanr as _spearmanr
    from statsmodels.stats.multitest import multipletests

    def fmt_p(p):
        if p is None or (isinstance(p, float) and np.isnan(p)):
            return "NA"
        return f"{p:.4f}" if p >= 1e-4 else f"{p:.2e}"

    def fmt_r(r):
        if r is None or (isinstance(r, float) and np.isnan(r)):
            return "NA"
        return f"{r:.3f}"

    def fmt_b(b):
        if b is None or (isinstance(b, float) and np.isnan(b)):
            return "NA"
        return f"{b:.3f}"

    def is_sig(p, alpha=0.05):
        return (p is not None) and (not (isinstance(p, float) and np.isnan(p))) and (p < alpha)

    def logistic_nonzero_assoc(df_, x_col, y_col):
        out = {"beta": np.nan, "p": np.nan}
        if len(df_) < min_points_panel:
            return out

        x = df_[x_col].astype(float)
        z = (df_[y_col] > 0).astype(int)

        if z.nunique() < 2 or x.nunique() < 2:
            return out

        try:
            X = sm.add_constant(x, has_constant="add")
            model = sm.Logit(z, X).fit(disp=False)
            out["beta"] = float(model.params[x_col])
            out["p"] = float(model.pvalues[x_col])
        except Exception:
            pass
        return out

    def positive_severity_assoc(df_, x_col, y_col):
        out = {"rho": np.nan, "p": np.nan, "n_pos": 0}
        pos = df_[df_[y_col] > 0].copy()
        out["n_pos"] = len(pos)

        if len(pos) < min_points_positive:
            return out
        if pos[x_col].nunique() < 2 or pos[y_col].nunique() < 2:
            return out

        rho, p = _spearmanr(pos[x_col], pos[y_col])
        out["rho"] = float(rho)
        out["p"] = float(p)
        return out

    # Reconstruct deconfounded person-level metric
    wo_df = settings[setting_name].copy().reset_index(drop=True)
    wo_ppl = setting_ppl_vars[setting_name]
    wo_resid_col = f"{wo_ppl}_person_resid"

    wo_df["sex_num"] = wo_df["sex"].map(
        {
            "M": 0, "F": 1,
            "m": 0, "f": 1,
            "male": 0, "female": 1,
            "Male": 0, "Female": 1,
        }
    )

    confounders = ["sex_num", "age", "education", f"{MODEL}_avg_words"]
    wo_resid = residualize_ppl(wo_df, wo_ppl, confounders)

    wo_df[wo_resid_col] = np.nan
    wo_df.loc[wo_resid.index, wo_resid_col] = wo_resid
    wo_patients = wo_df[wo_df["is_patient"] == 1].copy()

    clinical_targets = (
        merged_corr.loc[merged_corr["sig_fdr_resid"] == True, "Clinical Variable"]
        .dropna()
        .tolist()
    )
    clinical_targets = [c for c in clinical_targets if c in wo_patients.columns]

    print("FDR-significant residualized clinical targets:")
    print(clinical_targets)

    if len(clinical_targets) == 0:
        raise ValueError("No FDR-significant clinical targets available to plot.")

    nice = {
        "GAF": "GAF",
        "TLI_Disorganization": "TLI Disorganization",
        "verbal_IQ": "Verbal IQ",
        "SAPS_PositiveFormaleDenkstoerung_ENIGMA": "SAPS formal thought disorder",
        "SAPS_Sum_ENIGMA": "SAPS Sum",
        "YMRS_Sum": "YMRS Sum",
        "SANS_AbulieApathie_ENIGMA": "SANS abulia/apathy",
        "SAPS_Wahn_ENIGMA": "SAPS delusions",
        "SAPS_BizarresVerhalten_ENIGMA": "SAPS bizarre behavior",
        "LexicalVF": "Lexical verbal fluency",
    }

    panel_store = {}
    stats_rows = []

    # Pass 1: compute stats
    for cvar in clinical_targets:
        sub = wo_patients[[wo_resid_col, cvar, "diagnosis"]].dropna().copy()
        present_dx = [d for d in DIAG_ORDER if d != "HC" and d in sub["diagnosis"].values]

        overall_prev = logistic_nonzero_assoc(sub, wo_resid_col, cvar)
        overall_sev = positive_severity_assoc(sub, wo_resid_col, cvar)

        ov_row = {
            "measure": cvar,
            "measure_label": nice.get(cvar, cvar),
            "diagnosis": "Overall",
            "n": len(sub),
            "n_pos": int((sub[cvar] > 0).sum()),
            "non_zero_pct": float((sub[cvar] > 0).mean() * 100.0) if len(sub) else np.nan,
            "prev_beta": overall_prev["beta"],
            "prev_p": overall_prev["p"],
            "sev_rho_pos": overall_sev["rho"],
            "sev_p_pos": overall_sev["p"],
            "prev_p_fdr": np.nan,
            "sev_p_fdr": np.nan,
        }
        stats_rows.append(ov_row)

        by_dx_rows = []
        for dx in present_dx:
            s = sub[sub["diagnosis"] == dx].copy()
            dx_prev = logistic_nonzero_assoc(s, wo_resid_col, cvar)
            dx_sev = positive_severity_assoc(s, wo_resid_col, cvar)

            row = {
                "measure": cvar,
                "measure_label": nice.get(cvar, cvar),
                "diagnosis": dx,
                "n": len(s),
                "n_pos": int((s[cvar] > 0).sum()),
                "non_zero_pct": float((s[cvar] > 0).mean() * 100.0) if len(s) else np.nan,
                "prev_beta": dx_prev["beta"],
                "prev_p": dx_prev["p"],
                "sev_rho_pos": dx_sev["rho"],
                "sev_p_pos": dx_sev["p"],
                "prev_p_fdr": np.nan,
                "sev_p_fdr": np.nan,
            }
            stats_rows.append(row)
            by_dx_rows.append(row)

        panel_store[cvar] = {
            "sub": sub,
            "present_dx": present_dx,
            "overall_row": ov_row,
            "diag_rows": by_dx_rows,
        }

    df_stats = pd.DataFrame(stats_rows)

    # FDR for overall rows across measures
    overall_mask = df_stats["diagnosis"] == "Overall"
    prev_valid = overall_mask & df_stats["prev_p"].notna()
    sev_valid = overall_mask & df_stats["sev_p_pos"].notna()

    if prev_valid.sum() > 0:
        _, p_adj, _, _ = multipletests(
            df_stats.loc[prev_valid, "prev_p"].values,
            alpha=0.05,
            method="fdr_bh",
        )
        df_stats.loc[prev_valid, "prev_p_fdr"] = p_adj

    if sev_valid.sum() > 0:
        _, p_adj, _, _ = multipletests(
            df_stats.loc[sev_valid, "sev_p_pos"].values,
            alpha=0.05,
            method="fdr_bh",
        )
        df_stats.loc[sev_valid, "sev_p_fdr"] = p_adj

    # FDR by diagnosis across measures (keeps diagnosis-specific relevance)
    diag_levels = [d for d in df_stats["diagnosis"].dropna().unique() if d != "Overall"]
    for dx in diag_levels:
        m = df_stats["diagnosis"] == dx
        pv = m & df_stats["prev_p"].notna()
        sv = m & df_stats["sev_p_pos"].notna()

        if pv.sum() > 0:
            _, p_adj, _, _ = multipletests(
                df_stats.loc[pv, "prev_p"].values,
                alpha=0.05,
                method="fdr_bh",
            )
            df_stats.loc[pv, "prev_p_fdr"] = p_adj

        if sv.sum() > 0:
            _, p_adj, _, _ = multipletests(
                df_stats.loc[sv, "sev_p_pos"].values,
                alpha=0.05,
                method="fdr_bh",
            )
            df_stats.loc[sv, "sev_p_fdr"] = p_adj

    # Build caption text table (one row per measure)
    caption_rows = []
    for cvar in clinical_targets:
        rows_m = df_stats[df_stats["measure"] == cvar].copy()
        ov = rows_m[rows_m["diagnosis"] == "Overall"].iloc[0]
        diag_rows = rows_m[rows_m["diagnosis"] != "Overall"].copy()

        sig_prev_dx = diag_rows.loc[diag_rows["prev_p_fdr"] < 0.05, "diagnosis"].tolist()
        sig_sev_dx = diag_rows.loc[diag_rows["sev_p_fdr"] < 0.05, "diagnosis"].tolist()

        caption = (
            f"{nice.get(cvar, cvar)}: overall non-zero prevalence logit beta={fmt_b(ov['prev_beta'])}, "
            f"pFDR={fmt_p(ov['prev_p_fdr'])}; "
            f"overall positive-only severity rho={fmt_r(ov['sev_rho_pos'])}, "
            f"pFDR={fmt_p(ov['sev_p_fdr'])}; "
            f"N={int(ov['n'])}, N+={int(ov['n_pos'])} ({ov['non_zero_pct']:.1f}% non-zero). "
            f"Significant diagnoses (pFDR<0.05): Prev={','.join(sig_prev_dx) if sig_prev_dx else 'none'}, "
            f"Sev+={','.join(sig_sev_dx) if sig_sev_dx else 'none'}."
        )

        if include_diag_in_caption and len(diag_rows) > 0:
            parts = []
            for _, r in diag_rows.iterrows():
                parts.append(
                    f"{r['diagnosis']}: N={int(r['n'])}, N+={int(r['n_pos'])}, "
                    f"prev beta={fmt_b(r['prev_beta'])}, prev pFDR={fmt_p(r['prev_p_fdr'])}, "
                    f"sev rho={fmt_r(r['sev_rho_pos'])}, sev pFDR={fmt_p(r['sev_p_fdr'])}"
                )
            caption = caption + " By diagnosis -> " + "; ".join(parts) + "."

        caption_rows.append(
            {
                "measure": cvar,
                "measure_label": nice.get(cvar, cvar),
                "n_overall": int(ov["n"]),
                "n_pos_overall": int(ov["n_pos"]),
                "non_zero_pct_overall": float(ov["non_zero_pct"]),
                "overall_prev_beta": ov["prev_beta"],
                "overall_prev_p": ov["prev_p"],
                "overall_prev_p_fdr": ov["prev_p_fdr"],
                "overall_sev_rho_pos": ov["sev_rho_pos"],
                "overall_sev_p_pos": ov["sev_p_pos"],
                "overall_sev_p_fdr": ov["sev_p_fdr"],
                "sig_prev_dx": ",".join(sig_prev_dx) if len(sig_prev_dx) else "none",
                "sig_sev_dx": ",".join(sig_sev_dx) if len(sig_sev_dx) else "none",
                "caption_text": caption,
            }
        )

    df_caption = pd.DataFrame(caption_rows)

    # Save CSVs
    out_dir = os.path.join(FIGURE_DIR, "main_figures")
    os.makedirs(out_dir, exist_ok=True)

    detailed_csv = os.path.join(out_dir, f"{csv_prefix}_detailed.csv")
    caption_csv = os.path.join(out_dir, f"{csv_prefix}_caption_text.csv")

    df_stats.to_csv(detailed_csv, index=False)
    df_caption.to_csv(caption_csv, index=False)

    print(f"Saved detailed stats CSV: {detailed_csv}")
    print(f"Saved caption text CSV: {caption_csv}")

    # Pass 2: plot with top-space legend + compact diagnosis pFDR lines
    n = len(clinical_targets)
    ncols = 2
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(6.8 * ncols, 5.1 * nrows),
        squeeze=False,
    )

    stats_out = {}
    legend_handles = None
    legend_labels = None

    for i, cvar in enumerate(clinical_targets):
        ax = axes[i // ncols][i % ncols]
        pack = panel_store[cvar]
        sub = pack["sub"]
        present_dx = pack["present_dx"]

        if len(sub) < min_points_panel:
            ax.text(
                0.5,
                0.5,
                f"Not enough data for {nice.get(cvar, cvar)}",
                transform=ax.transAxes,
                ha="center",
                va="center",
            )
            ax.axis("off")
            continue

        # Plot points and diagnosis lines
        for dx in present_dx:
            s = sub[sub["diagnosis"] == dx].copy()
            color = DIAG_COLORS.get(dx, "#888888")

            zeros = s[s[cvar] == 0]
            pos = s[s[cvar] > 0]

            if len(zeros) > 0:
                ax.scatter(
                    zeros[wo_resid_col],
                    zeros[cvar],
                    facecolors="none",
                    edgecolors=color,
                    alpha=0.55,
                    s=26,
                    linewidths=0.8,
                )

            if len(pos) > 0:
                ax.scatter(
                    pos[wo_resid_col],
                    pos[cvar],
                    color=color,
                    label=dx,
                    alpha=0.60,
                    s=30,
                    edgecolors="white",
                    linewidths=0.25,
                )

            if (
                len(pos) >= min_points_line
                and pos[wo_resid_col].nunique() > 1
                and pos[cvar].nunique() > 1
            ):
                z_dx = np.polyfit(pos[wo_resid_col], pos[cvar], 1)
                xs_dx = np.linspace(pos[wo_resid_col].min(), pos[wo_resid_col].max(), 100)
                ax.plot(xs_dx, np.polyval(z_dx, xs_dx), color=color, linewidth=1.8, alpha=0.9)

        # Overall positive-only line
        pos_all = sub[sub[cvar] > 0].copy()
        if (
            len(pos_all) >= min_points_line
            and pos_all[wo_resid_col].nunique() > 1
            and pos_all[cvar].nunique() > 1
        ):
            z_all = np.polyfit(pos_all[wo_resid_col], pos_all[cvar], 1)
            xs_all = np.linspace(pos_all[wo_resid_col].min(), pos_all[wo_resid_col].max(), 120)
            ax.plot(
                xs_all,
                np.polyval(z_all, xs_all),
                color="black",
                linestyle="--",
                linewidth=2.0,
                alpha=0.85,
                label="Overall y>0 fit",
            )

        # Compact stats text: overall pFDR + diagnosis-level pFDR (P and S) with stars
        ov = df_caption[df_caption["measure"] == cvar].iloc[0]
        diag_rows = (
            df_stats[(df_stats["measure"] == cvar) & (df_stats["diagnosis"] != "Overall")]
            .copy()
            .set_index("diagnosis")
        )

        dx_parts = []
        for dx in present_dx:
            if dx not in diag_rows.index:
                continue
            prev_pf = diag_rows.loc[dx, "prev_p_fdr"]
            sev_pf = diag_rows.loc[dx, "sev_p_fdr"]
            p_star = "*" if is_sig(prev_pf) else ""
            s_star = "*" if is_sig(sev_pf) else ""
            dx_parts.append(f"{dx} P:{fmt_p(prev_pf)}{p_star} S:{fmt_p(sev_pf)}{s_star}")

        if len(dx_parts) == 0:
            dx_text = "Dx pFDR none"
        else:
            split = 2
            dx_lines = [" | ".join(dx_parts[k:k + split]) for k in range(0, len(dx_parts), split)]
            dx_text = "\n".join(dx_lines)

        panel_text = (
            f"N+={int(ov['n_pos_overall'])}/{int(ov['n_overall'])} ({ov['non_zero_pct_overall']:.1f}%)\n"
            f"Overall P:{fmt_p(ov['overall_prev_p_fdr'])}  S:{fmt_p(ov['overall_sev_p_fdr'])}\n"
            f"{dx_text}"
        )

        ax.text(
            0.02,
            0.98,
            panel_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.0,
            bbox=dict(boxstyle="round,pad=0.22", facecolor="white", alpha=0.74, edgecolor="0.75"),
        )

        ax.set_xlabel("Person-level deconfounded length-adjusted LM loss", fontsize=10)
        ax.set_ylabel(nice.get(cvar, cvar), fontsize=10)
        ax.set_title(nice.get(cvar, cvar), fontsize=11)
        ax.grid(alpha=0.2)

        # Save handles once for a figure-level legend in top margin
        if legend_handles is None:
            h, l = ax.get_legend_handles_labels()
            if len(h) > 0:
                legend_handles, legend_labels = h, l

        stats_out[cvar] = {
            "overall": df_stats[
                (df_stats["measure"] == cvar) & (df_stats["diagnosis"] == "Overall")
            ].to_dict("records")[0],
            "by_diagnosis": df_stats[
                (df_stats["measure"] == cvar) & (df_stats["diagnosis"] != "Overall")
            ].to_dict("records"),
        }

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(
        "FDR-significant clinical associations: hurdle-style prevalence + severity",
        fontsize=14,
        y=0.99,
    )

    if legend_handles is not None and legend_labels is not None:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.955),
            ncol=min(5, max(2, len(legend_labels))),
            fontsize=8,
            frameon=True,
            framealpha=0.9,
        )

    # Extra top space so legend does not overlap points
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    fig.text(
        0.01,
        0.005,
        "P=prevalence pFDR, S=severity(y>0) pFDR, *=pFDR<0.05",
        ha="left",
        va="bottom",
        fontsize=8,
        color="0.35",
    )

    save_path = os.path.join(out_dir, save_name)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {save_path}")
    plt.show()

    return fig, stats_out, df_stats, df_caption

In [ ]:
fig, stats_out, df_stats, df_caption = plot_fdr_significant_clinical_scatter_grid_with_dx_lines(
    settings=settings,
    setting_ppl_vars=setting_ppl_vars,
    merged_corr=merged_corr,
    MODEL=MODEL,
    FIGURE_DIR=FIGURE_DIR,
    residualize_ppl=residualize_ppl,
    DIAG_ORDER=DIAG_ORDER,
    DIAG_COLORS=DIAG_COLORS,
    setting_name="with_outliers",
)

In [ ]:
"""
Three complementary tests for transdiagnostic vs. diagnosis-driven effects.

1. Partial correlation controlling for diagnosis
   → "Does PPL↔clinical survive after removing diagnostic group membership?"

2. Within-diagnosis centering
   → "If we remove ALL between-diagnosis variance, is there still a within-group signal?"

3. OLS interaction test
   → "Does the slope (PPL → clinical) differ across diagnoses?"

Together these tell the full story:
- Partial r significant + interaction ns → transdiagnostic, homogeneous effect
- Partial r significant + interaction sig → transdiagnostic but heterogeneous
- Partial r ns → between-group artifact
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests


# ── 1. Partial Spearman correlation controlling for diagnosis ─────
def partial_spearman_controlling_diagnosis(
    df, ppl_col, cvar, dx_col="diagnosis"
):
    """
    Rank-based partial correlation:
    Regress ranks(PPL) and ranks(cvar) each on diagnosis dummies,
    then correlate the residuals.
    """
    sub = df[[ppl_col, cvar, dx_col]].dropna().copy()
    if len(sub) < 15 or sub[dx_col].nunique() < 2:
        return {"r_partial": np.nan, "p_partial": np.nan, "n": len(sub)}

    # Rank transform both variables
    sub["rank_ppl"] = rankdata(sub[ppl_col])
    sub["rank_cvar"] = rankdata(sub[cvar])

    # Diagnosis dummies
    dx_dummies = pd.get_dummies(sub[dx_col], drop_first=True, dtype=float)
    X_dx = sm.add_constant(dx_dummies, has_constant="add")

    # Residualize both ranked variables on diagnosis
    resid_ppl = sm.OLS(sub["rank_ppl"], X_dx).fit().resid
    resid_cvar = sm.OLS(sub["rank_cvar"], X_dx).fit().resid

    # Correlate the residuals → partial Spearman
    r, p = spearmanr(resid_ppl, resid_cvar)

    return {"r_partial": r, "p_partial": p, "n": len(sub)}


# ── 2. Within-diagnosis centering ─────────────────────────────────
def within_diagnosis_correlation(
    df, ppl_col, cvar, dx_col="diagnosis"
):
    """
    Center PPL and cvar within each diagnosis group,
    then correlate. This removes ALL between-group variance.
    Pure within-group signal only.
    """
    sub = df[[ppl_col, cvar, dx_col]].dropna().copy()
    if len(sub) < 15 or sub[dx_col].nunique() < 2:
        return {"r_within": np.nan, "p_within": np.nan, "n": len(sub)}

    # Group-mean centering
    sub["ppl_centered"] = sub.groupby(dx_col)[ppl_col].transform(
        lambda x: x - x.mean()
    )
    sub["cvar_centered"] = sub.groupby(dx_col)[cvar].transform(
        lambda x: x - x.mean()
    )

    r, p = spearmanr(sub["ppl_centered"], sub["cvar_centered"])

    return {"r_within": r, "p_within": p, "n": len(sub)}


# ── 3. OLS interaction test ───────────────────────────────────────
def diagnosis_interaction_test(
    df, ppl_col, cvar, dx_col="diagnosis"
):
    """
    OLS: cvar ~ PPL + dx_dummies + PPL × dx_dummies
    F-test on the interaction terms.
    Non-significant → slopes are homogeneous across diagnoses.
    """
    sub = df[[ppl_col, cvar, dx_col]].dropna().copy().reset_index(drop=True)
    if len(sub) < 20 or sub[dx_col].nunique() < 2:
        return {
            "F_interaction": np.nan,
            "p_interaction": np.nan,
            "n": len(sub),
        }

    dx_dummies = pd.get_dummies(sub[dx_col], drop_first=True, prefix="dx", dtype=float)
    dx_dummies.index = sub.index

    interaction_cols = []
    for col in dx_dummies.columns:
        icol = f"int_{col}"
        sub[icol] = dx_dummies[col].values * sub[ppl_col].values
        interaction_cols.append(icol)

    X = pd.DataFrame({"const": 1.0, "ppl": sub[ppl_col].values}, index=sub.index)
    for col in dx_dummies.columns:
        X[col] = dx_dummies[col].values
    for icol in interaction_cols:
        X[icol] = sub[icol].values

    y = sub[cvar].values

    try:
        model = sm.OLS(y, X).fit()
        r_matrix = np.zeros((len(interaction_cols), X.shape[1]))
        param_names = list(model.params.index)
        for i, ic in enumerate(interaction_cols):
            r_matrix[i, param_names.index(ic)] = 1
        f_test = model.f_test(r_matrix)
        return {
            "F_interaction": float(f_test.fvalue),
            "p_interaction": float(f_test.pvalue),
            "n": len(sub),
        }
    except Exception as e:
        print(f"  Interaction test failed for {cvar}: {e}")
        return {
            "F_interaction": np.nan,
            "p_interaction": np.nan,
            "n": len(sub),
        }


# ── Run all three for a list of clinical variables ────────────────
def transdiagnostic_analysis(
    df, ppl_col, clinical_vars, dx_col="diagnosis"
):
    """
    Run all three tests for each clinical variable.
    Returns a DataFrame with columns for each test + FDR correction.
    """
    rows = []

    for cvar in clinical_vars:
        if cvar not in df.columns:
            continue

        partial = partial_spearman_controlling_diagnosis(df, ppl_col, cvar, dx_col)
        within = within_diagnosis_correlation(df, ppl_col, cvar, dx_col)
        interaction = diagnosis_interaction_test(df, ppl_col, cvar, dx_col)

        # Also compute the naive (unadjusted) overall correlation for reference
        sub = df[[ppl_col, cvar]].dropna()
        if len(sub) >= 5:
            r_naive, p_naive = spearmanr(sub[ppl_col], sub[cvar])
        else:
            r_naive, p_naive = np.nan, np.nan

        rows.append({
            "clinical_var": cvar,
            "n": partial["n"],
            # Naive overall
            "r_overall": r_naive,
            "p_overall": p_naive,
            # Partial (controlling for dx)
            "r_partial": partial["r_partial"],
            "p_partial": partial["p_partial"],
            # Within-dx centering
            "r_within": within["r_within"],
            "p_within": within["p_within"],
            # Interaction
            "F_interaction": interaction["F_interaction"],
            "p_interaction": interaction["p_interaction"],
        })

    result = pd.DataFrame(rows)

    # FDR across variables for partial and within tests
    for col in ["p_partial", "p_within", "p_interaction"]:
        valid = result[col].notna()
        if valid.sum() > 0:
            _, p_adj, _, _ = multipletests(
                result.loc[valid, col], alpha=0.05, method="fdr_bh"
            )
            result.loc[valid, f"{col}_fdr"] = p_adj

    # Interpretation column
    def interpret(row):
        partial_sig = row.get("p_partial_fdr", np.nan) < 0.05
        within_sig = row.get("p_within_fdr", np.nan) < 0.05
        interaction_sig = row.get("p_interaction_fdr", np.nan) < 0.05

        if partial_sig and not interaction_sig:
            return "transdiagnostic (homogeneous)"
        elif partial_sig and interaction_sig:
            return "transdiagnostic (heterogeneous)"
        elif not partial_sig and not within_sig:
            return "between-group artifact"
        elif within_sig and not partial_sig:
            return "weak within-group signal"
        else:
            return "ambiguous"

    result["interpretation"] = result.apply(interpret, axis=1)

    return result

In [ ]:
clinical_targets = merged_corr.loc[
    merged_corr["sig_fdr_resid"] == True, "Clinical Variable"
].tolist()

trans_df = transdiagnostic_analysis(
    df=wo_patients,
    ppl_col=wo_resid_col,
    clinical_vars=clinical_targets,
    dx_col="diagnosis",
)

# Key columns to inspect
display(trans_df[[
    "clinical_var", "n",
    "r_overall", "r_partial", "p_partial_fdr",
    "r_within", "p_within_fdr",
    "p_interaction_fdr", "interpretation",
]])

In [ ]:
"""
Main figure: FDR-significant clinical associations (overall).

Uses pre-computed FDR-corrected statistics from merged_corr.
Points colored by diagnosis for visual inspection.
Includes a formal diagnosis × PPL interaction test per panel.
"""

from scipy.stats import spearmanr as _spearmanr
from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
import statsmodels.api as sm
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ── Interaction test ──────────────────────────────────────────────
def diagnosis_interaction_test(df, ppl_col, cvar, dx_col="diagnosis"):
    """
    OLS: cvar ~ ppl + diagnosis_dummies + ppl × diagnosis_dummies
    Returns F-test p-value for the interaction terms.
    Non-significant p → effect is transdiagnostic.
    """
    sub = df[[ppl_col, cvar, dx_col]].dropna().copy().reset_index(drop=True)
    if len(sub) < 20 or sub[dx_col].nunique() < 2:
        return np.nan, np.nan

    # Build design matrix with simple column names
    dx_dummies = pd.get_dummies(sub[dx_col], drop_first=True, prefix="dx", dtype=float)
    dx_dummies.index = sub.index  # ensure alignment

    interaction_cols = []
    for col in dx_dummies.columns:
        icol = f"int_{col}"
        sub[icol] = dx_dummies[col].values * sub[ppl_col].values
        interaction_cols.append(icol)

    # Use simple "ppl" name to avoid special chars in column names
    X = pd.DataFrame({"const": 1.0, "ppl": sub[ppl_col].values}, index=sub.index)
    for col in dx_dummies.columns:
        X[col] = dx_dummies[col].values
    for icol in interaction_cols:
        X[icol] = sub[icol].values

    y = sub[cvar].values

    try:
        model = sm.OLS(y, X).fit()

        # F-test: restrict to interaction columns only
        r_matrix = np.zeros((len(interaction_cols), X.shape[1]))
        param_names = list(model.params.index)
        for i, ic in enumerate(interaction_cols):
            r_matrix[i, param_names.index(ic)] = 1

        f_test = model.f_test(r_matrix)
        return float(f_test.fvalue), float(f_test.pvalue)
    except Exception as e:
        print(f"  Interaction test failed for {cvar}: {e}")
        return np.nan, np.nan


# ── Main plotting function ────────────────────────────────────────
def plot_fdr_clinical_main(
    settings,
    setting_ppl_vars,
    merged_corr,
    MODEL,
    FIGURE_DIR,
    residualize_ppl,
    DIAG_ORDER,
    DIAG_COLORS,
    setting_name="without_outliers",
    save_name="fdr_clinical_main.png",
):
    nice = {
        "GAF": "GAF",
        "TLI_Disorganization": "TLI Disorganization",
        "verbal_IQ": "Verbal IQ",
        "SAPS_PositiveFormaleDenkstoerung_ENIGMA": "SAPS Positive Formal Thought Disorder",
        "SAPS_Sum_ENIGMA": "SAPS Sum",
        "YMRS_Sum": "YMRS Sum",
        "SANS_AbulieApathie_ENIGMA": "SANS Abulia/Apathy",
        "SAPS_Wahn_ENIGMA": "SAPS Delusions",
        "SAPS_BizarresVerhalten_ENIGMA": "SAPS Bizarre Behavior",
        "LexicalVF": "Lexical Verbal Fluency",
    }

    def fmt_p(p):
        if p is None or (isinstance(p, float) and np.isnan(p)):
            return "NA"
        return f"{p:.4f}" if p >= 1e-4 else f"{p:.2e}"

    # ── Prepare data ──────────────────────────────────────────────
    wo_df = settings[setting_name].copy().reset_index(drop=True)
    wo_ppl = setting_ppl_vars[setting_name]
    wo_resid_col = f"{wo_ppl}_person_resid"

    wo_df["sex_num"] = wo_df["sex"].map({
        "M": 0, "F": 1, "m": 0, "f": 1,
        "male": 0, "female": 1, "Male": 0, "Female": 1,
    })

    confounders = ["sex_num", "age", "education", f"{MODEL}_avg_words"]
    wo_resid = residualize_ppl(wo_df, wo_ppl, confounders)
    wo_df[wo_resid_col] = np.nan
    wo_df.loc[wo_resid.index, wo_resid_col] = wo_resid

    wo_patients = wo_df[wo_df["is_patient"] == 1].copy()

    # ── Select variables: use the ALREADY FDR-corrected gate ──────
    clinical_targets = (
        merged_corr.loc[merged_corr["sig_fdr_resid"] == True, "Clinical Variable"]
        .dropna()
        .tolist()
    )
    clinical_targets = [c for c in clinical_targets if c in wo_patients.columns]

    if len(clinical_targets) == 0:
        raise ValueError("No FDR-significant clinical targets to plot.")

    print(f"Plotting {len(clinical_targets)} FDR-significant variables (from merged_corr):")
    print(clinical_targets)

    # ── Pull pre-computed stats from merged_corr ──────────────────
    mc = merged_corr.set_index("Clinical Variable")

    # ── Plot ──────────────────────────────────────────────────────
    n = len(clinical_targets)
    ncols = 2
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(6.5 * ncols, 5.0 * nrows),
        squeeze=False,
    )

    interaction_rows = []

    for i, cvar in enumerate(clinical_targets):
        ax = axes[i // ncols][i % ncols]
        sub = wo_patients[[wo_resid_col, cvar, "diagnosis"]].dropna()

        if len(sub) < 5:
            ax.text(0.5, 0.5, f"Insufficient data: {nice.get(cvar, cvar)}",
                    transform=ax.transAxes, ha="center", va="center")
            ax.axis("off")
            continue

        # ── Scatter by diagnosis ──────────────────────────────────
        for dx in [d for d in DIAG_ORDER if d != "HC" and d in sub["diagnosis"].values]:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(
                s[wo_resid_col], s[cvar],
                color=DIAG_COLORS.get(dx, "#888"),
                label=dx, alpha=0.60, s=32,
                edgecolors="white", linewidths=0.3,
            )

        # ── Overall regression line ───────────────────────────────
        r, p = _spearmanr(sub[wo_resid_col], sub[cvar])

        if sub[wo_resid_col].nunique() > 1:
            z = np.polyfit(sub[wo_resid_col], sub[cvar], 1)
            xs = np.linspace(sub[wo_resid_col].min(), sub[wo_resid_col].max(), 100)
            ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.7, linewidth=1.5)

        # ── Interaction test ──────────────────────────────────────
        f_int, p_int = diagnosis_interaction_test(sub, wo_resid_col, cvar)
        interaction_rows.append({
            "clinical_var": cvar,
            "F_interaction": f_int,
            "p_interaction": p_int,
            "transdiagnostic": p_int >= 0.05 if not np.isnan(p_int) else None,
        })

        # ── Annotation: use merged_corr FDR stats ────────────────
        # Get the pre-computed rho and pFDR from the upstream analysis
        if cvar in mc.index:
            rho_fdr = mc.loc[cvar, "r_resid"]
            p_fdr = mc.loc[cvar, "p_fdr_resid"]
        else:
            rho_fdr, p_fdr = r, p  # fallback

        int_label = "transdiag." if (not np.isnan(p_int) and p_int >= 0.05) else "dx-specific"
        int_str = f"interaction p={fmt_p(p_int)} → {int_label}"

        ax.set_xlabel("Deconfounded length-adjusted LM loss", fontsize=10)
        ax.set_ylabel(nice.get(cvar, cvar), fontsize=10)
        ax.set_title(
            f"{nice.get(cvar, cvar)}\n"
            f"ρ = {rho_fdr:.3f}, p$_{{FDR}}$ = {fmt_p(p_fdr)}, N = {len(sub)}",
            fontsize=11,
        )

        # Small annotation for interaction test
        ax.text(
            0.02, 0.02, int_str,
            transform=ax.transAxes, ha="left", va="bottom",
            fontsize=8, color="0.4",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      alpha=0.7, edgecolor="0.75"),
        )

        ax.grid(alpha=0.2)

        if i == 0:
            ax.legend(fontsize=8, framealpha=0.8)

    # ── Turn off empty panels ─────────────────────────────────────
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(
        "Clinical associations with deconfounded LM loss (FDR-corrected)",
        fontsize=14, y=1.005,
    )
    fig.tight_layout()

    out_dir = os.path.join(FIGURE_DIR, "main_figures")
    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, save_name)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.show()

    # ── Print interaction summary ─────────────────────────────────
    int_df = pd.DataFrame(interaction_rows)
    print("\nDiagnosis × PPL interaction tests:")
    print(int_df.to_string(index=False))

    # Save interaction results
    int_df.to_csv(os.path.join(out_dir, "interaction_tests.csv"), index=False)

    return fig, int_df

In [ ]:
fig, interaction_df = plot_fdr_clinical_main(
    settings=settings,
    setting_ppl_vars=setting_ppl_vars,
    merged_corr=merged_corr,
    MODEL=MODEL,
    FIGURE_DIR=FIGURE_DIR,
    residualize_ppl=residualize_ppl,
    DIAG_ORDER=DIAG_ORDER,
    DIAG_COLORS=DIAG_COLORS,
    setting_name="with_outliers",
)

In [ ]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

def clinical_correlations_by_diagnosis(
    df,
    metric_col,
    clinical_vars,
    diagnosis_col="diagnosis",
    diagnoses=None,
    min_n=10,
):
    """
    Compute Spearman correlations between metric_col and clinical variables
    separately within each diagnosis group.
    """
    rows = []

    if diagnoses is None:
        diagnoses = sorted(df[diagnosis_col].dropna().unique())

    for dx in diagnoses:
        dx_df = df[df[diagnosis_col] == dx].copy()

        for cvar in clinical_vars:
            if cvar not in dx_df.columns:
                continue

            sub = dx_df[[metric_col, cvar]].replace([np.inf, -np.inf], np.nan).dropna()

            if len(sub) < min_n:
                rows.append({
                    "diagnosis": dx,
                    "clinical_variable": cvar,
                    "n": len(sub),
                    "rho": np.nan,
                    "p": np.nan,
                    "note": "too few observations",
                })
                continue

            # If one variable is constant, Spearman is undefined
            if sub[metric_col].nunique() < 2 or sub[cvar].nunique() < 2:
                rows.append({
                    "diagnosis": dx,
                    "clinical_variable": cvar,
                    "n": len(sub),
                    "rho": np.nan,
                    "p": np.nan,
                    "note": "constant variable",
                })
                continue

            rho, p = spearmanr(sub[metric_col], sub[cvar])

            rows.append({
                "diagnosis": dx,
                "clinical_variable": cvar,
                "n": len(sub),
                "rho": rho,
                "p": p,
                "note": "",
            })

    return pd.DataFrame(rows)

In [ ]:
metric_col = wo_resid_col  # e.g. qwen_loss_resid_median_wins_person_resid

clinical_targets = (
    merged_corr.loc[merged_corr["sig_fdr_resid"] == True, "Clinical Variable"]
    .dropna()
    .tolist()
)

by_dx_corr = clinical_correlations_by_diagnosis(
    df=wo_patients,
    metric_col=metric_col,
    clinical_vars=clinical_targets,
    diagnosis_col="diagnosis",
    diagnoses=["BD", "MDD", "SSD"],
    min_n=10,
)

by_dx_corr

In [ ]:
by_dx_corr_pivot = by_dx_corr.pivot(
    index="clinical_variable",
    columns="diagnosis",
    values="rho",
)

by_dx_corr_pivot

In [ ]:
by_dx_p_pivot = by_dx_corr.pivot(
    index="clinical_variable",
    columns="diagnosis",
    values="p",
)

by_dx_p_pivot

In [ ]:
# Descriptive zero-prevalence table — no statistical testing, no FDR problem
zero_inflated_vars = [
    "SAPS_Sum_ENIGMA", "YMRS_Sum", "SAPS_Wahn_ENIGMA",
    "SAPS_BizarresVerhalten_ENIGMA", "SANS_AbulieApathie_ENIGMA",
    "SAPS_PositiveFormaleDenkstoerung_ENIGMA", "TLI_Disorganization",
]

rows = []
for cvar in zero_inflated_vars:
    if cvar not in wo_patients.columns:
        continue
    sub = wo_patients[[wo_resid_col, cvar]].dropna()
    n_total = len(sub)
    n_zero = (sub[cvar] == 0).sum()
    n_pos = n_total - n_zero
    
    # Overall Spearman (all data, including zeros)
    r_all, p_all = spearmanr(sub[wo_resid_col], sub[cvar])
    
    # Spearman on positive-only (descriptive, not tested for significance)
    pos = sub[sub[cvar] > 0]
    if len(pos) >= 10:
        r_pos, _ = spearmanr(pos[wo_resid_col], pos[cvar])
    else:
        r_pos = np.nan
    
    rows.append({
        "variable": cvar,
        "N": n_total,
        "N_zero": n_zero,
        "pct_zero": round(n_zero / n_total * 100, 1),
        "rho_all": round(r_all, 3),
        "rho_positive_only": round(r_pos, 3) if not np.isnan(r_pos) else "NA",
    })

zero_table = pd.DataFrame(rows)
print(zero_table.to_string(index=False))

In [ ]:
def classify_pattern(row):
    if row["pct_zero"] < 20:
        return "continuous (minimal floor effect)"
    r_all = abs(row["rho_all"])
    r_pos = row["rho_positive_only"]
    if isinstance(r_pos, str):
        return "insufficient positive cases"
    r_pos = abs(r_pos)
    if r_pos > r_all * 1.2:
        return "severity-driven"
    elif r_pos < r_all * 0.5:
        return "prevalence-driven"
    else:
        return "mixed"

zero_table["pattern"] = zero_table.apply(classify_pattern, axis=1)
print(zero_table[["variable", "N", "pct_zero", "rho_all", 
                   "rho_positive_only", "pattern"]].to_string(index=False))

In [ ]:
from scipy.stats import pointbiserialr

rows_hurdle = []
for cvar in zero_inflated_vars:
    if cvar not in wo_patients.columns:
        continue
    sub = wo_patients[[wo_resid_col, cvar]].dropna()
    n_total = len(sub)
    n_zero = (sub[cvar] == 0).sum()
    n_pos = n_total - n_zero

    r_all, p_all = spearmanr(sub[wo_resid_col], sub[cvar])

    binary = (sub[cvar] > 0).astype(int)
    if binary.nunique() == 2:
        r_prev, p_prev = pointbiserialr(binary, sub[wo_resid_col])
    else:
        r_prev, p_prev = np.nan, np.nan

    pos = sub[sub[cvar] > 0]
    if len(pos) >= 10 and pos[cvar].nunique() >= 3:
        r_sev, p_sev = spearmanr(pos[wo_resid_col], pos[cvar])
    else:
        r_sev, p_sev = np.nan, np.nan

    rows_hurdle.append({
        "variable": cvar,
        "N": n_total,
        "pct_zero": round(n_zero / n_total * 100, 1),
        "n_pos": n_pos,
        "rho_overall": round(r_all, 3),
        "r_prevalence": round(r_prev, 3),
        "p_prevalence": p_prev,
        "rho_severity": round(r_sev, 3) if not np.isnan(r_sev) else np.nan,
        "p_severity": p_sev,
    })

hurdle_df = pd.DataFrame(rows_hurdle)

# FDR within each part separately
from statsmodels.stats.multitest import multipletests
for col in ["p_prevalence", "p_severity"]:
    valid = hurdle_df[col].notna()
    if valid.sum() > 1:
        _, p_adj, _, _ = multipletests(
            hurdle_df.loc[valid, col], alpha=0.05, method="fdr_bh"
        )
        hurdle_df.loc[valid, f"{col}_fdr"] = p_adj

print(hurdle_df.to_string(index=False))

In [ ]:
""" # Clean table without misleading labels
report_df = hurdle_df[[
    "variable", "N", "pct_zero", "n_pos",
    "rho_overall",
    "r_prevalence", "p_prevalence_fdr",
    "rho_severity", "p_severity_fdr",
]].copy()

# Add a note column instead of a binary classification
def note(row):
    parts = []
    if row["p_prevalence_fdr"] < 0.05:
        parts.append("prevalence*")
    if row["p_severity_fdr"] < 0.05:
        parts.append("severity*")
    if not parts:
        if row["n_pos"] < 40:
            parts.append(f"severity underpowered (n={row['n_pos']})")
        else:
            parts.append("ns after FDR")
    return "; ".join(parts)

#report_df["note"] = report_df.apply(note, axis=1)
print(report_df.to_string(index=False)) """

In [ ]:
from statsmodels.stats.multitest import multipletests

def add_fdr_by_diagnosis(corr_df):
    out = corr_df.copy()
    out["p_fdr"] = np.nan
    out["sig_fdr"] = False

    for dx, idx in out.groupby("diagnosis").groups.items():
        mask = out.loc[idx, "p"].notna()
        valid_idx = out.loc[idx].loc[mask].index

        if len(valid_idx) == 0:
            continue

        reject, p_fdr, _, _ = multipletests(
            out.loc[valid_idx, "p"],
            alpha=0.05,
            method="fdr_bh",
        )

        out.loc[valid_idx, "p_fdr"] = p_fdr
        out.loc[valid_idx, "sig_fdr"] = reject

    return out

by_dx_corr = add_fdr_by_diagnosis(by_dx_corr)
by_dx_corr.sort_values(["diagnosis", "p_fdr"])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os

rho_mat = by_dx_corr.pivot(
    index="clinical_variable",
    columns="diagnosis",
    values="rho",
)

p_fdr_mat = by_dx_corr.pivot(
    index="clinical_variable",
    columns="diagnosis",
    values="p_fdr",
)

annot = rho_mat.copy().astype(object)

for row in rho_mat.index:
    for col in rho_mat.columns:
        rho = rho_mat.loc[row, col]
        p_fdr = p_fdr_mat.loc[row, col]

        if pd.isna(rho):
            annot.loc[row, col] = ""
        else:
            star = "***" if p_fdr < 0.001 else "**" if p_fdr < 0.01 else "*" if p_fdr < 0.05 else ""
            annot.loc[row, col] = f"{rho:.2f}{star}"

plt.figure(figsize=(7, max(4, 0.45 * len(rho_mat))))
sns.heatmap(
    rho_mat.astype(float),
    annot=annot,
    fmt="",
    cmap="vlag",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Spearman ρ"},
)

plt.title("Within-diagnosis correlations with deconfounded length-adjusted LM loss")
plt.xlabel("Diagnosis")
plt.ylabel("Clinical variable")
plt.tight_layout()

save_path = os.path.join(
    FIGURE_DIR,
    "main_figures",
    "within_diagnosis_clinical_correlations_heatmap.png",
)
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {save_path}")

## 9. Summary

In [ ]:
print("Analysis complete.")
print(f"Figures saved to: {os.path.abspath(FIGURE_DIR)}")

10. Variability analysis

In [ ]:
from scipy.stats import iqr as _iqr, spearmanr
import numpy as np
import pandas as pd

# ── 1. Compute within-subject variability from sentence-level data ──

# Use the residualized sentence-level loss (length-adjusted)
var_agg = (
    sentence_df_resid
    .groupby("file_id")["avg_loss_length_resid"]
    .agg(
        ppl_sd="std",
        ppl_iqr=lambda x: _iqr(x, nan_policy="omit"),
        ppl_cv=lambda x: x.std() / x.mean() if x.mean() != 0 else np.nan,
        ppl_range=lambda x: x.max() - x.min(),
        n_sentences="count",
    )
    .reset_index()
)

# Rename with model prefix
var_agg = var_agg.rename(columns={
    "ppl_sd": f"{MODEL}_ppl_sd_within",
    "ppl_iqr": f"{MODEL}_ppl_iqr_within",
    "ppl_cv": f"{MODEL}_ppl_cv_within",
    "ppl_range": f"{MODEL}_ppl_range_within",
})

# Drop n_sentences if already in df
var_agg = var_agg.drop(columns=["n_sentences"], errors="ignore")

# Merge into person-level df
new_cols = [c for c in var_agg.columns if c == "file_id" or c not in df.columns]
df = df.merge(var_agg[new_cols], on="file_id", how="left")

# Also merge into settings
for sname in settings:
    settings[sname] = settings[sname].merge(
        var_agg[new_cols], on="file_id", how="left"
    )

print(f"Added variability columns: {[c for c in var_agg.columns if c != 'file_id']}")
print(f"Sample values:\n{df[[f'{MODEL}_ppl_sd_within', f'{MODEL}_ppl_iqr_within']].describe()}")

# ── 2. Group comparison: do patients have more variable perplexity? ──

var_metrics = [
    f"{MODEL}_ppl_sd_within",
    f"{MODEL}_ppl_iqr_within",
    f"{MODEL}_ppl_cv_within",
]

print("Within-subject perplexity variability: HC vs Patient")
print("-" * 70)
for vm in var_metrics:
    comp = group_comparison(df, vm)
    print(f"{vm}:")
    print(f"  HC:  {df.loc[df['is_patient']==0, vm].median():.4f}")
    print(f"  PAT: {df.loc[df['is_patient']==1, vm].median():.4f}")
    print(f"  p(MW)={comp['p(MW)']:.4f}, d={comp['Cohens d']:.3f}")
    print()

# ── 3. Clinical correlations with variability metrics ──

patients = df[df["is_patient"] == 1].copy()

rows = []
for vm in var_metrics:
    for cvar in clinical_vars_corr:
        sub = patients[[vm, cvar]].dropna()
        if len(sub) < 10:
            continue
        r, p = spearmanr(sub[vm], sub[cvar])
        rows.append({
            "variability_metric": vm,
            "clinical_var": cvar,
            "n": len(sub),
            "rho": round(r, 3),
            "p": p,
        })

var_corr = pd.DataFrame(rows)

# FDR within each metric
from statsmodels.stats.multitest import multipletests

var_corr["p_fdr"] = np.nan
for vm in var_metrics:
    mask = var_corr["variability_metric"] == vm
    valid = mask & var_corr["p"].notna()
    if valid.sum() > 1:
        _, p_adj, _, _ = multipletests(
            var_corr.loc[valid, "p"], alpha=0.05, method="fdr_bh"
        )
        var_corr.loc[valid, "p_fdr"] = p_adj

var_corr["sig_fdr"] = var_corr["p_fdr"] < 0.05

print("\nSignificant variability-clinical associations (pFDR < 0.05):")
sig = var_corr[var_corr["sig_fdr"] == True]
if len(sig) > 0:
    print(sig.to_string(index=False))
else:
    print("None survived FDR")

print("\nTop 10 by uncorrected p:")
print(var_corr.nsmallest(10, "p").to_string(index=False))


# ── 4. Does variability add signal beyond the median? ──
# Partial correlation: variability → clinical, controlling for median PPL

from functools import partial

def partial_corr_controlling_median(patients, var_metric, cvar, median_metric=PPL_VAR):
    """Spearman partial correlation of variability with clinical var,
    controlling for median perplexity."""
    sub = patients[[var_metric, cvar, median_metric]].dropna()
    if len(sub) < 15:
        return {"r_partial": np.nan, "p_partial": np.nan, "n": len(sub)}
    
    from scipy.stats import rankdata
    
    # Rank everything
    r_var = rankdata(sub[var_metric])
    r_clin = rankdata(sub[cvar])
    r_med = rankdata(sub[median_metric])
    
    # Residualize both on median PPL ranks
    import statsmodels.api as sm
    X = sm.add_constant(r_med)
    resid_var = sm.OLS(r_var, X).fit().resid
    resid_clin = sm.OLS(r_clin, X).fit().resid
    
    r, p = spearmanr(resid_var, resid_clin)
    return {"r_partial": r, "p_partial": p, "n": len(sub)}

# Test: does IQR predict clinical vars beyond what median already captures?
iqr_metric = f"{MODEL}_ppl_iqr_within"
rows_partial = []
for cvar in clinical_vars_corr:
    res = partial_corr_controlling_median(patients, iqr_metric, cvar)
    res["clinical_var"] = cvar
    rows_partial.append(res)

partial_df = pd.DataFrame(rows_partial)
valid = partial_df["p_partial"].notna()
if valid.sum() > 1:
    _, p_adj, _, _ = multipletests(
        partial_df.loc[valid, "p_partial"], alpha=0.05, method="fdr_bh"
    )
    partial_df.loc[valid, "p_partial_fdr"] = p_adj

print("\nIQR variability → clinical (controlling for median PPL):")
print(partial_df.nsmallest(10, "p_partial").to_string(index=False))

In [ ]:
iqr_col = f"{MODEL}_ppl_iqr_within"
iqr_resid_col = f"{iqr_col}_resid"
confounders = ["sex_num", "age", "education", f"{MODEL}_avg_words"]
""" df_all = df_deconf.copy()  # ← this already has RESID_VAR
df_all["sex_num"] = df_all["sex"].map({
    "M": 0, "F": 1, "m": 0, "f": 1,
    "male": 0, "female": 1, "Male": 0, "Female": 1,
})

# Merge IQR if not already present
if iqr_col not in df_all.columns:
    df_all = df_all.merge(var_agg[["file_id", iqr_col]], on="file_id", how="left")

iqr_resid = residualize_ppl(df_all, iqr_col, confounders)
df_all[iqr_resid_col] = np.nan
df_all.loc[iqr_resid.index, iqr_resid_col] = iqr_resid

patients_deconf = df_all[df_all["is_patient"] == 1].copy()

# Re-run partial correlations with DECONFOUNDED IQR
# controlling for DECONFOUNDED median
rows_partial = []
for cvar in clinical_vars_corr:
    res = partial_corr_controlling_median(
        patients_deconf, 
        iqr_resid_col, 
        cvar, 
        median_metric=RESID_VAR,  # deconfounded median
    )
    res["clinical_var"] = cvar
    rows_partial.append(res)

partial_df_deconf = pd.DataFrame(rows_partial)
valid = partial_df_deconf["p_partial"].notna()
if valid.sum() > 1:
    _, p_adj, _, _ = multipletests(
        partial_df_deconf.loc[valid, "p_partial"], alpha=0.05, method="fdr_bh"
    )
    partial_df_deconf.loc[valid, "p_partial_fdr"] = p_adj

print("Deconfounded IQR → clinical (controlling for deconfounded median):")
print(partial_df_deconf.nsmallest(10, "p_partial").to_string(index=False)) """

In [ ]:
from scipy.stats import spearmanr
def bootstrap_spearman_ci(x, y, n_boot=5000, ci=0.95):
    rhos = []
    for _ in range(n_boot):
        idx = np.random.choice(len(x), len(x), replace=True)
        r, _ = spearmanr(x[idx], y[idx])
        rhos.append(r)
    lower = np.percentile(rhos, (1-ci)/2 * 100)
    upper = np.percentile(rhos, (1+ci)/2 * 100)
    return lower, upper

In [ ]:
# ── Bootstrap CIs for key reported results ────────────────────────
from scipy.stats import spearmanr
np.random.seed(42)
N_BOOT = 5000

# 1. CI for Cohen's d (group comparison)
def bootstrap_cohens_d_ci(df, ppl_col, n_boot=N_BOOT, ci=0.95):
    hc = df.loc[df["is_patient"] == 0, ppl_col].dropna().values
    pat = df.loc[df["is_patient"] == 1, ppl_col].dropna().values
    ds = []
    for _ in range(n_boot):
        hc_b = np.random.choice(hc, len(hc), replace=True)
        pat_b = np.random.choice(pat, len(pat), replace=True)
        pooled_std = np.sqrt(
            ((len(hc_b)-1)*hc_b.std(ddof=1)**2 + (len(pat_b)-1)*pat_b.std(ddof=1)**2)
            / (len(hc_b) + len(pat_b) - 2)
        )
        d = (pat_b.mean() - hc_b.mean()) / pooled_std if pooled_std > 0 else 0
        ds.append(d)
    alpha = (1 - ci) / 2
    return np.percentile(ds, alpha * 100), np.percentile(ds, (1 - alpha) * 100)

d_lo, d_hi = bootstrap_cohens_d_ci(df_deconf, PPL_VAR)
d_lo_r, d_hi_r = bootstrap_cohens_d_ci(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
print(f"Cohen's d CI (raw):          [{d_lo:.3f}, {d_hi:.3f}]")
print(f"Cohen's d CI (residualized): [{d_lo_r:.3f}, {d_hi_r:.3f}]")

# 2. CIs for FDR-significant clinical correlations
def bootstrap_spearman_ci(x, y, n_boot=N_BOOT, ci=0.95):
    x, y = np.array(x), np.array(y)
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    rhos = []
    for _ in range(n_boot):
        idx = np.random.choice(len(x), len(x), replace=True)
        r, _ = spearmanr(x[idx], y[idx])
        rhos.append(r)
    alpha = (1 - ci) / 2
    return np.percentile(rhos, alpha * 100), np.percentile(rhos, (1 - alpha) * 100)

patients_ci = df_deconf[df_deconf["is_patient"] == 1]
sig_vars = merged_corr.loc[merged_corr["sig_fdr_resid"] == True, "Clinical Variable"].tolist()

ci_rows = []
for cvar in sig_vars:
    if cvar not in patients_ci.columns:
        continue
    lo, hi = bootstrap_spearman_ci(
        patients_ci[RESID_VAR].values, 
        patients_ci[cvar].values,
    )
    r = merged_corr.set_index("Clinical Variable").loc[cvar, "r_resid"]
    ci_rows.append({
        "clinical_var": cvar,
        "rho": round(r, 3),
        "CI_lo": round(lo, 3),
        "CI_hi": round(hi, 3),
    })

ci_df = pd.DataFrame(ci_rows)
print("\nBootstrap 95% CIs for FDR-significant clinical correlations:")
print(ci_df.to_string(index=False))

In [ ]:
hc = df_deconf[df_deconf["is_patient"] == 0]
r_hc, p_hc = spearmanr(
    hc[f"{MODEL}_avg_words"].dropna(), 
    hc[PPL_VAR].dropna()
)
print(f"HC only: avg_words ↔ PPL: r={r_hc:.3f}, p={p_hc:.4f}")

In [ ]:
# Check overlap of confounder distributions
for conf in ["age", "education", f"{MODEL}_avg_words"]:
    hc_range = df_deconf.loc[df_deconf["is_patient"]==0, conf].dropna()
    pat_range = df_deconf.loc[df_deconf["is_patient"]==1, conf].dropna()
    pat_outside = ((pat_range < hc_range.min()) | (pat_range > hc_range.max())).sum()
    print(f"{conf}: HC range [{hc_range.min():.1f}, {hc_range.max():.1f}], "
          f"patients outside: {pat_outside}/{len(pat_range)}")

In [ ]:
# If you go with rank-biserial r instead:
np.random.seed(42)

for dx in ["BD", "MDD", "SSD"]:
    hc = df_deconf.loc[df_deconf["is_patient"] == 0, RESID_VAR].dropna().values
    pat = df_deconf.loc[df_deconf["diagnosis"] == dx, RESID_VAR].dropna().values
    
    rs = []
    for _ in range(5000):
        hc_b = np.random.choice(hc, len(hc), replace=True)
        pat_b = np.random.choice(pat, len(pat), replace=True)
        u, _ = mannwhitneyu(pat_b, hc_b, alternative="two-sided")
        r = 1 - (2 * u) / (len(hc_b) * len(pat_b))
        rs.append(r)
    
    lo, hi = np.percentile(rs, [2.5, 97.5])
    print(f"{dx}: r_rb = {np.median(rs):.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
# ── Does LM loss add signal beyond sentence length? ──────────────
import statsmodels.api as sm
from scipy.stats import spearmanr

RAW_METRIC = f"{MODEL}_loss_median"  # raw loss, no length adjustment
LENGTH_METRIC = f"{MODEL}_avg_words"

patients = df[df["is_patient"] == 1].copy()

# 1. Partial correlation: LM loss → clinical, controlling for length
from scipy.stats import rankdata

def partial_spearman(df, x_col, y_col, control_col):
    sub = df[[x_col, y_col, control_col]].dropna()
    if len(sub) < 15:
        return np.nan, np.nan, len(sub)
    
    rx = rankdata(sub[x_col])
    ry = rankdata(sub[y_col])
    rc = rankdata(sub[control_col])
    
    X = sm.add_constant(rc)
    resid_x = sm.OLS(rx, X).fit().resid
    resid_y = sm.OLS(ry, X).fit().resid
    
    r, p = spearmanr(resid_x, resid_y)
    return r, p, len(sub)

print("Incremental validity: LM loss controlling for sentence length")
print("=" * 75)
print(f"{'Clinical var':<45} {'r_raw':>7} {'r_partial':>10} {'p_partial':>10}")
print("-" * 75)

for cvar in clinical_vars_corr:
    if cvar not in patients.columns:
        continue
    sub = patients[[RAW_METRIC, cvar]].dropna()
    r_raw, _ = spearmanr(sub[RAW_METRIC], sub[cvar])
    r_part, p_part, n = partial_spearman(patients, RAW_METRIC, cvar, LENGTH_METRIC)
    sig = "***" if p_part < .001 else "**" if p_part < .01 else "*" if p_part < .05 else ""
    print(f"{cvar:<45} {r_raw:>+.3f} {r_part:>+.3f} {p_part:>10.4f} {sig}")

# 2. HC vs Patient: does LM loss discriminate beyond length?
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

analysis_df = df[[RAW_METRIC, LENGTH_METRIC, "is_patient"]].dropna()

X_length = analysis_df[[LENGTH_METRIC]].values
X_loss = analysis_df[[RAW_METRIC]].values
X_both = analysis_df[[RAW_METRIC, LENGTH_METRIC]].values
y = analysis_df["is_patient"].values

auc_length = cross_val_score(
    LogisticRegression(), X_length, y, cv=10, scoring="roc_auc"
).mean()
auc_loss = cross_val_score(
    LogisticRegression(), X_loss, y, cv=10, scoring="roc_auc"
).mean()
auc_both = cross_val_score(
    LogisticRegression(), X_both, y, cv=10, scoring="roc_auc"
).mean()

print(f"\nGroup discrimination (10-fold CV AUC):")
print(f"  Sentence length only:  {auc_length:.3f}")
print(f"  LM loss only:          {auc_loss:.3f}")
print(f"  Both combined:         {auc_both:.3f}")
print(f"  LM loss increment:     {auc_both - auc_length:+.3f}")

In [ ]:
# 3. Formal likelihood ratio test: does adding LM loss improve the model?
full_df = df[[RAW_METRIC, LENGTH_METRIC, "is_patient"]].dropna()

# Restricted model: group ~ length only
X_r = sm.add_constant(full_df[[LENGTH_METRIC]])
model_r = sm.Logit(full_df["is_patient"], X_r).fit(disp=False)

# Full model: group ~ length + LM loss
X_f = sm.add_constant(full_df[[LENGTH_METRIC, RAW_METRIC]])
model_f = sm.Logit(full_df["is_patient"], X_f).fit(disp=False)

# Likelihood ratio test
lr_stat = -2 * (model_r.llf - model_f.llf)
from scipy.stats import chi2
lr_p = chi2.sf(lr_stat, df=1)

print(f"\nLikelihood ratio test (adding LM loss to length-only model):")
print(f"  χ² = {lr_stat:.3f}, p = {lr_p:.4f}")
print(f"  Length-only AIC: {model_r.aic:.1f}")
print(f"  Length+Loss AIC: {model_f.aic:.1f}")
print(f"  Full model LM loss β = {model_f.params[RAW_METRIC]:.3f}, "
      f"p = {model_f.pvalues[RAW_METRIC]:.4f}")

In [ ]:
# Incremental validity: is LM loss beyond length driven by SSD only?

# 1. Likelihood ratio test WITHOUT SSD
no_ssd = df[df["diagnosis"] != "SSD"].copy()
no_ssd_full = no_ssd[[RAW_METRIC, LENGTH_METRIC, "is_patient"]].dropna()

X_r = sm.add_constant(no_ssd_full[[LENGTH_METRIC]])
model_r = sm.Logit(no_ssd_full["is_patient"], X_r).fit(disp=False)

X_f = sm.add_constant(no_ssd_full[[LENGTH_METRIC, RAW_METRIC]])
model_f = sm.Logit(no_ssd_full["is_patient"], X_f).fit(disp=False)

lr_stat = -2 * (model_r.llf - model_f.llf)
lr_p = chi2.sf(lr_stat, df=1)
print(f"WITHOUT SSD: χ² = {lr_stat:.3f}, p = {lr_p:.4f}")
print(f"  LM loss β = {model_f.params[RAW_METRIC]:.3f}, p = {model_f.pvalues[RAW_METRIC]:.4f}")

# 2. Partial correlations by diagnosis
print("\nPartial correlations (LM loss | length) by diagnosis:")
print(f"{'Diagnosis':<8} {'Clinical var':<40} {'r_partial':>10} {'p':>10}")
print("-" * 72)

key_vars = ["YMRS_Sum", "SAPS_Sum_ENIGMA", "verbal_IQ", 
            "TLI_Disorganization", "LexicalVF", "SANS_AbulieApathie_ENIGMA"]

for dx in ["BD", "MDD", "SSD"]:
    pat_dx = patients[patients["diagnosis"] == dx]
    for cvar in key_vars:
        if cvar not in pat_dx.columns:
            continue
        r_part, p_part, n = partial_spearman(pat_dx, RAW_METRIC, cvar, LENGTH_METRIC)
        if np.isnan(r_part):
            continue
        sig = "*" if p_part < .05 else ""
        print(f"{dx:<8} {cvar:<40} {r_part:>+.3f} {p_part:>10.4f} {sig}  (n={n})")
    print()

In [ ]:
# ── Ablation summary: three levels of deconfounding ───────────────

# Level 1: Raw loss
raw_comp = group_comparison(df, f"{MODEL}_loss_median_wins")
raw_corr = clinical_correlations(
    df[df["is_patient"]==1], f"{MODEL}_loss_median_wins", clinical_vars_corr
)

# Level 2: Raw loss, demographics-only residualized
DECONF_DEMO = ["sex_num", "age", "education"]
df_demo = df.copy()
demo_resid = residualize_ppl(df_demo, f"{MODEL}_loss_median_wins", DECONF_DEMO)
df_demo["loss_demo_resid"] = np.nan
df_demo.loc[demo_resid.index, "loss_demo_resid"] = demo_resid

demo_comp = group_comparison(df_demo.dropna(subset=["loss_demo_resid"]), "loss_demo_resid")
demo_corr = clinical_correlations(
    df_demo[df_demo["is_patient"]==1].dropna(subset=["loss_demo_resid"]),
    "loss_demo_resid", clinical_vars_corr
)

# Level 3: Length-residualized loss + full deconfounding (your existing pipeline)
# This is already in your RESID_VAR from the qwen_loss_resid_median_wins analysis
# Re-use those results
# Level 3: Length-residualized loss + full deconfounding
LENRESID_VAR = f"{MODEL}_loss_resid_median_wins"  # sentence-level length-adjusted
FULL_DECONF_VARS = ["sex_num", "age", "education", f"{MODEL}_avg_words"]

# Check it exists in df
if LENRESID_VAR not in df_deconf.columns:
    # Merge from loss_person_df
    if LENRESID_VAR in loss_person_df.columns:
        df_deconf = df_deconf.merge(
            loss_person_df[["file_id", LENRESID_VAR]], on="file_id", how="left"
        )
    else:
        raise ValueError(f"{LENRESID_VAR} not found")

# Person-level residualize the length-adjusted metric
LENRESID_DECONF = f"{LENRESID_VAR}_resid"
full_resid = residualize_ppl(df_deconf, LENRESID_VAR, FULL_DECONF_VARS)
df_deconf[LENRESID_DECONF] = np.nan
df_deconf.loc[full_resid.index, LENRESID_DECONF] = full_resid

full_comp = group_comparison(
    df_deconf.dropna(subset=[LENRESID_DECONF]), LENRESID_DECONF
)
full_corr = clinical_correlations(
    df_deconf[df_deconf["is_patient"]==1].dropna(subset=[LENRESID_DECONF]),
    LENRESID_DECONF, clinical_vars_corr
)
full_comp = group_comparison(
    df_deconf.dropna(subset=[f"{MODEL}_loss_resid_median_wins_resid"]),
    f"{MODEL}_loss_resid_median_wins_resid"
)
full_corr = clinical_correlations(
    df_deconf[df_deconf["is_patient"]==1].dropna(subset=[f"{MODEL}_loss_resid_median_wins_resid"]),
    f"{MODEL}_loss_resid_median_wins_resid", clinical_vars_corr
)

# ── Summary table ─────────────────────────────────────────────────
ablation_summary = pd.DataFrame([
    {
        "Level": "1. Raw LM loss",
        "Sentence-level": "none",
        "Person-level": "none",
        "d": raw_comp["Cohens d"],
        "p(MW)": raw_comp["p(MW)"],
        "FDR-sig correlations": (raw_corr["p_fdr"] < 0.05).sum(),
        "Total tested": len(raw_corr),
    },
    {
        "Level": "2. Raw + demographics",
        "Sentence-level": "none",
        "Person-level": "age, sex, education",
        "d": demo_comp["Cohens d"],
        "p(MW)": demo_comp["p(MW)"],
        "FDR-sig correlations": (demo_corr["p_fdr"] < 0.05).sum(),
        "Total tested": len(demo_corr),
    },
    {
        "Level": "3. Length-adjusted + full",
        "Sentence-level": "log(word count)",
        "Person-level": "age, sex, education, avg_words",
        "d": full_comp["Cohens d"],
        "p(MW)": full_comp["p(MW)"],
        "FDR-sig correlations": (full_corr["p_fdr"] < 0.05).sum(),
        "Total tested": len(full_corr),
    },
])

print(ablation_summary.to_string(index=False))

# Save
ablation_summary.to_csv(
    os.path.join(FIGURE_DIR, "advanced", "deconfounding_ablation.csv"), 
    index=False
)

In [ ]:
# ── Ablation: Clinical correlations across three levels ───────────

# Merge all three correlation tables
corr_ablation = raw_corr[["Clinical Variable", "r", "p_fdr", "sig_fdr"]].rename(
    columns={"r": "ρ_raw", "p_fdr": "pFDR_raw", "sig_fdr": "sig_raw"}
).merge(
    demo_corr[["Clinical Variable", "r", "p_fdr", "sig_fdr"]].rename(
        columns={"r": "ρ_demo", "p_fdr": "pFDR_demo", "sig_fdr": "sig_demo"}
    ), on="Clinical Variable"
).merge(
    full_corr[["Clinical Variable", "r", "p_fdr", "sig_fdr"]].rename(
        columns={"r": "ρ_full", "p_fdr": "pFDR_full", "sig_fdr": "sig_full"}
    ), on="Clinical Variable"
)

# Sort by raw p
corr_ablation = corr_ablation.sort_values("pFDR_raw")

print("Clinical correlations across deconfounding levels:")
print(corr_ablation.to_string(index=False))

corr_ablation.to_csv(
    os.path.join(FIGURE_DIR, "advanced", "clinical_correlations_ablation.csv"),
    index=False
)


# ── Ablation: Per-diagnosis group comparison across three levels ──

from functions.analysis.statistics import pairwise_vs_hc

# Level 1: raw
pw_raw = pairwise_vs_hc(df, f"{MODEL}_loss_median_wins")

# Level 2: demographics-only
pw_demo = pairwise_vs_hc(
    df_demo.dropna(subset=["loss_demo_resid"]), "loss_demo_resid"
)

# Level 3: length-adjusted + full
pw_full = pairwise_vs_hc(
    df_deconf.dropna(subset=[f"{MODEL}_loss_resid_median_wins_resid"]),
    f"{MODEL}_loss_resid_median_wins_resid"
)

pw_raw_df = pd.DataFrame(pw_raw)
pw_demo_df = pd.DataFrame(pw_demo)
pw_full_df = pd.DataFrame(pw_full)

dx_ablation = pw_raw_df[["Diagnosis", "n"]].copy()
dx_ablation["d_raw"] = pw_raw_df["Cohen's d"]
dx_ablation["p_raw"] = pw_raw_df["p"]
dx_ablation["d_demo"] = pw_demo_df["Cohen's d"]
dx_ablation["p_demo"] = pw_demo_df["p"]
dx_ablation["d_full"] = pw_full_df["Cohen's d"]
dx_ablation["p_full"] = pw_full_df["p"]

# Add significance stars
for col in ["p_raw", "p_demo", "p_full"]:
    dx_ablation[col.replace("p_", "sig_")] = dx_ablation[col].apply(
        lambda x: "***" if x < .001 else "**" if x < .01 else "*" if x < .05 else "ns"
    )

print("\nPer-diagnosis vs HC across deconfounding levels:")
print(dx_ablation.to_string(index=False))

dx_ablation.to_csv(
    os.path.join(FIGURE_DIR, "advanced", "per_diagnosis_ablation.csv"),
    index=False
)

In [ ]:
# ── Deconfounding validation ──────────────────────────────────────
from scipy.stats import spearmanr

# Level 2: demographics-only (your primary sensitivity)
DECONF_DEMO = ["sex_num", "age", "education"]

print("Deconfounding validation: demographics-only")
print(f"{'Confounder':<15} {'ρ (raw)':>10} {'p (raw)':>10} {'ρ (resid)':>10} {'p (resid)':>10}")
print("-" * 60)
for conf in DECONF_DEMO:
    sub = df_demo[[conf, f"{MODEL}_loss_median_wins", "loss_demo_resid"]].dropna()
    r_before, p_before = spearmanr(sub[conf], sub[f"{MODEL}_loss_median_wins"])
    r_after, p_after = spearmanr(sub[conf], sub["loss_demo_resid"])
    print(f"{conf:<15} {r_before:>+.4f} {p_before:>10.4f} {r_after:>+.4f} {p_after:>10.4f}")

# Level 3: full deconfounding
FULL_VARS = ["sex_num", "age", "education"]

print("\nDeconfounding validation: full (demographics + length)")
print(f"{'Confounder':<20} {'ρ (raw)':>10} {'p (raw)':>10} {'ρ (resid)':>10} {'p (resid)':>10}")
print("-" * 65)
for conf in FULL_VARS:
    sub = df_deconf[[conf, f"{MODEL}_loss_median_wins", LENRESID_DECONF]].dropna()
    r_before, p_before = spearmanr(sub[conf], sub[f"{MODEL}_loss_median_wins"])
    r_after, p_after = spearmanr(sub[conf], sub[LENRESID_DECONF])
    print(f"{conf:<20} {r_before:>+.4f} {p_before:>10.4f} {r_after:>+.4f} {p_after:>10.4f}")

In [ ]:
sub = df[[f"{MODEL}_avg_words", f"{MODEL}_loss_median_wins", LENRESID_VAR]].dropna()

r_raw, p_raw = spearmanr(sub[f"{MODEL}_avg_words"], sub[f"{MODEL}_loss_median_wins"])
r_adj, p_adj = spearmanr(sub[f"{MODEL}_avg_words"], sub[LENRESID_VAR])

print(f"avg_words ↔ raw loss:             ρ = {r_raw:+.4f}, p = {p_raw:.4f}")
print(f"avg_words ↔ length-adjusted loss: ρ = {r_adj:+.4f}, p = {p_adj:.4f}")

In [ ]:
# Is sex correlated with the length-adjusted metric but not the raw one?
for metric, label in [
    (f"{MODEL}_loss_median_wins", "Raw loss"),
    (f"{MODEL}_loss_resid_median_wins", "Length-adjusted loss"),
]:
    if metric not in df.columns:
        continue
    sub = df[["sex_num", metric]].dropna()
    r, p = spearmanr(sub["sex_num"], sub[metric])
    print(f"{label}: sex ↔ metric  ρ = {r:.4f}, p = {p:.4f}")

# Check if residualize_ppl actually receives sex_num without NaN
print(f"\nsex_num NaN count: {df_deconf['sex_num'].isna().sum()}")
print(f"sex_num in confounders: {'sex_num' in FULL_DECONF_VARS}")

In [ ]:
# ── Level 3: Length-adjusted + demographics-only deconfounding ────
# Length is already handled at the sentence level.
# avg_words is NOT included at person level because conditioning on it
# creates a collider artifact with sex (sentence-level residualization
# reverses the length-PPL sign from ρ=-0.50 to ρ=+0.35, and avg_words
# is a shared cause of sex and the overcorrected metric).

LENRESID_VAR = f"{MODEL}_loss_resid_median_wins"
LENRESID_DECONF = f"{LENRESID_VAR}_resid"
LEVEL3_DECONF = ["sex_num", "age", "education"]

# Ensure length-adjusted metric exists in df_deconf
if LENRESID_VAR not in df_deconf.columns:
    df_deconf = df_deconf.merge(
        loss_person_df[["file_id", LENRESID_VAR]], on="file_id", how="left"
    )

# Person-level residualization: demographics only
full_resid = residualize_ppl(df_deconf, LENRESID_VAR, LEVEL3_DECONF)
df_deconf[LENRESID_DECONF] = np.nan
df_deconf.loc[full_resid.index, LENRESID_DECONF] = full_resid

# Validate
print("Level 3 validation (demographics only, length handled at sentence level):")
for conf in LEVEL3_DECONF + [f"{MODEL}_avg_words"]:
    sub = df_deconf[[conf, LENRESID_DECONF]].dropna()
    r, p = spearmanr(sub[conf], sub[LENRESID_DECONF])
    status = "controlled" if conf in LEVEL3_DECONF else "sentence-level only"
    print(f"  {conf:<20} ρ = {r:+.4f}, p = {p:.4f}  ({status})")

# Group comparison + clinical correlations
full_comp = group_comparison(
    df_deconf.dropna(subset=[LENRESID_DECONF]), LENRESID_DECONF
)
full_corr = clinical_correlations(
    df_deconf[df_deconf["is_patient"]==1].dropna(subset=[LENRESID_DECONF]),
    LENRESID_DECONF, clinical_vars_corr
)

# Per-diagnosis
pw_full = pairwise_vs_hc(
    df_deconf.dropna(subset=[LENRESID_DECONF]), LENRESID_DECONF
)

In [ ]:
full_comp = group_comparison(
    df_deconf.dropna(subset=[LENRESID_DECONF]), LENRESID_DECONF
)
full_corr = clinical_correlations(
    df_deconf[df_deconf["is_patient"]==1].dropna(subset=[LENRESID_DECONF]),
    LENRESID_DECONF, clinical_vars_corr
)

In [ ]:
full_comp

In [ ]:
full_corr